# HM-RAG + MAMMQA: repaired 16-question pipeline-validation smoke test

**Experiment:** `qa_smoke16_pipeline_repair_v2_seed42`  
**Scope:** objective implementation repair only; no gold-answer tuning and no architecture redesign.

## Failed-run audit

- Cell 17 `plan_query`/subquery extraction stringified dictionaries; `retrieve_evidence` then omitted the original question in 13/16 planned query sets. Fixed with typed normalization, deduplication, query assertions, and original-question-first records.
- Cell 18 `route_budgets` used planner priority to reduce candidate depth. Fixed so soft routing searches every available modality at K=20 and affects only context priority.
- Cell 16 `llm_json`/Qwen tokenization right-truncated a complete 1,792-token chat, corrupting Stage II and verifier instructions. Replaced with deterministic evidence-only budgeting at 4,096 tokens and `truncation=False`.
- Cells 19--22 silently converted malformed structured generations into empty dictionaries/`INSUFFICIENT`. Replaced with raw/parsed/parse-success envelopes and one format-only retry.
- Cell 22 treated a missing verifier status as unsupported and forced abstention. Verifier errors are now infrastructure errors and aggregation continues; only valid conflict/insufficient can trigger retry/abstention.
- Cell 20 gave every Stage-II anchor all experts and all raw evidence. Each anchor now receives its own Stage-I output plus raw non-anchor evidence only.
- Cell 23 mixed the proposed verifier into `MAMMQA_ORACLE`. The repaired oracle is gold evidence -> shared Stage I -> shared Stage II -> shared aggregator only.
- Cells 10/18 retain the existing dense, BM25, image/CLIP, RRF, graph, and context-selection algorithms; only provenance and candidate/context separation were repaired. Cell 24 retains shared EM/F1 normalization.

This repair freezes the exact failed-run qids, searches every modality at candidate K=20, budgets only evidence
before chat construction, uses one format-repair retry, distinguishes verifier infrastructure errors from
epistemic insufficiency, and reports pipeline health before QA metrics.

In [ ]:
# Cell 1 - environment diagnostics; allocator configured before importing torch
import importlib.util, os, platform, subprocess, sys
os.environ.setdefault("PYTORCH_ALLOC_CONF","expandable_segments:True")
print("Python:",sys.version)
try:
    import torch
    print("PyTorch:",torch.__version__,"CUDA:",torch.cuda.is_available(),"GPUs:",torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):print(i,torch.cuda.get_device_name(i))
except Exception as e:print("PyTorch diagnostic failed:",e)

In [ ]:
# Cell 2 - install only missing, released packages
REQUIRED = {
    "transformers": "transformers>=4.49,<5", "accelerate": "accelerate>=1,<2",
    "bitsandbytes": "bitsandbytes>=0.45,<1", "qwen_vl_utils": "qwen-vl-utils>=0.0.8",
    "sentence_transformers": "sentence-transformers>=3,<6", "faiss": "faiss-cpu>=1.8,<2",
    "networkx": "networkx>=3,<4", "pandas": "pandas>=2,<3", "tabulate": "tabulate>=0.9,<1",
    "rank_bm25": "rank-bm25>=0.2.2,<1",
}
missing = [spec for module, spec in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else: print("All optional dependencies are already installed.")

In [ ]:
# Cell 3 - frozen repair configuration, imports and isolated experiment paths
import ast, copy, csv, gc, hashlib, json, math, os, pickle, random, re, string, time, zipfile
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Sequence, Tuple
import faiss, networkx as nx, numpy as np, pandas as pd, torch
from PIL import Image
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

SEED=42; RUN_MODE="smoke"; RUN_RETRIEVAL_ONLY=False
EXPERIMENT_ID="qa_smoke16_pipeline_repair_v2_seed42"
FAILED_EXPERIMENT_ID="qa_smoke16_full_pipeline_seed42"
MODEL_NAME="Qwen/Qwen2.5-VL-7B-Instruct"
EMBED_MODEL="BAAI/bge-small-en-v1.5"; CLIP_MODEL="openai/clip-vit-base-patch32"
RETRIEVAL_MODE="hybrid"; ROUTER_MODE="soft"
RETRIEVAL_CANDIDATE_K=20; MIN_METRIC_CANDIDATE_K=10; FALLBACK_RETRIEVAL_K=20
TEXT_CONTEXT_K=5; TABLE_CONTEXT_K=3; IMAGE_CONTEXT_K=3
FALLBACK_TEXT_K=2; FALLBACK_TABLE_K=1; FALLBACK_IMAGE_K=1
ROUTER_CONFIDENCE_THRESHOLD=0.65; PLANNER_MODALITY_THRESHOLD=0.55
RRF_K=60; GRAPH_SEED_K=10; GRAPH_MAX_EXPANDED_DOCS=50; GRAPH_MAX_DEGREE=50
TEXT_TOP_K=TEXT_CONTEXT_K; TABLE_TOP_K=TABLE_CONTEXT_K; IMAGE_TOP_K=IMAGE_CONTEXT_K; IMAGE_CANDIDATES=50
MAX_RETRIEVAL_ROUNDS=2; ENABLE_GRAPH=True; ENABLE_WEB=False
IMAGE_RETRIEVAL_MODE="hybrid"; AGGREGATOR_SEES_QUESTION=True
TABLE_CHUNK_ROWS=50; LARGE_TABLE_ROWS=120; IMAGE_REASONING_USES_TITLE=False

# Engineering budgets are fixed before the rerun and are not selected from answer quality.
MAX_INPUT_TOKENS=4096; MAX_LLM_NEW_TOKENS=256; FORMAT_REPAIR_NEW_TOKENS=128
STAGE_MAX_NEW_TOKENS={"planner":128,"stage1":160,"stage2":192,"verifier":128,"aggregator":96}
MAMMQA_TEMPERATURE=0.3; MAMMQA_TOP_P=0.7; MAMMQA_DO_SAMPLE=True
PLANNER_DO_SAMPLE=False; VERIFIER_DO_SAMPLE=False; IMAGE_PROCESSOR_USE_FAST=False
RUN_ABLATIONS=False; CLIP_DEVICE="cpu"

BASE_OUTPUT_ROOT=Path("/kaggle/working/hm_mammqa")
OUTPUT_ROOT=BASE_OUTPUT_ROOT/EXPERIMENT_ID
FAILED_OUTPUT_ROOT=BASE_OUTPUT_ROOT/FAILED_EXPERIMENT_ID
DIRS={
    "cache":BASE_OUTPUT_ROOT/"cache","indexes":BASE_OUTPUT_ROOT/"indexes","images":BASE_OUTPUT_ROOT/"images",
    "predictions":OUTPUT_ROOT/"predictions","metrics":OUTPUT_ROOT/"metrics","logs":OUTPUT_ROOT/"logs",
    "diagnostics":OUTPUT_ROOT/"diagnostics","offload":OUTPUT_ROOT/"offload",
}
DIRS["retrieval_logs"]=DIRS["diagnostics"]/"retrieval_logs"; DIRS["ablations"]=DIRS["diagnostics"]/"ablations"
for p in set(DIRS.values()):p.mkdir(parents=True,exist_ok=True)
GENERATION_DIAGNOSTICS_PATH=DIRS["logs"]/"generation_diagnostics.jsonl"
PLANNER_DIAGNOSTICS_PATH=DIRS["logs"]/"planner_diagnostics.jsonl"
TRUNCATION_DIAGNOSTICS_PATH=DIRS["logs"]/"truncation_diagnostics.jsonl"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
assert SEED==42 and RUN_MODE=="smoke" and RUN_RETRIEVAL_ONLY is False
assert RETRIEVAL_CANDIDATE_K>=MIN_METRIC_CANDIDATE_K>=10
assert MAX_RETRIEVAL_ROUNDS==2 and not ENABLE_WEB and IMAGE_REASONING_USES_TITLE is False

def gpu_memory(label):
    if not torch.cuda.is_available():print(label,"CPU only"); return
    stats=[]
    for i in range(torch.cuda.device_count()):
        free,total=torch.cuda.mem_get_info(i); stats.append(f"GPU{i} free={free/2**30:.2f}/{total/2**30:.2f}GiB")
    print(label," | ".join(stats))
gpu_memory("startup:")

In [ ]:
# Cell 4 - confirmed MMQA paths with unambiguous fallback
DATA_ROOT=Path("/kaggle/input/datasets/adibraian/multimodalqa-dataset")
NAMES={
 "train":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_train.jsonl",
 "dev":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_dev.jsonl",
 "test":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_test.jsonl",
 "texts":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_texts.jsonl",
 "tables":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_tables.jsonl",
 "images":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_images.jsonl"}
FOLDERS={k:f"MMQA_{k}.jsonl" for k in NAMES}
def resolve_dataset_file(key:str)->Path:
    primary=DATA_ROOT/FOLDERS[key]/NAMES[key]
    if primary.is_file(): return primary
    roots=[DATA_ROOT,Path("/kaggle/input")]
    matches=sorted({p.resolve() for root in roots if root.exists() for p in root.rglob(NAMES[key])})
    if len(matches)!=1: raise FileNotFoundError(f"{key}: expected one {NAMES[key]}, found {matches}")
    return matches[0]
PATHS={k:resolve_dataset_file(k) for k in NAMES}
for k,p in PATHS.items(): print(f"{k:7s} {p}")

In [ ]:
# Cell 5 - JSONL loading
def load_jsonl(path:Path)->List[dict]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for no,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except json.JSONDecodeError as e: raise ValueError(f"{path}:{no}: {e}") from e
    return out
dev_raw=load_jsonl(PATHS["dev"]); texts_raw=load_jsonl(PATHS["texts"])
tables_raw=load_jsonl(PATHS["tables"]); images_raw=load_jsonl(PATHS["images"])
print(len(dev_raw),len(texts_raw),len(tables_raw),len(images_raw))

In [ ]:
# Cell 6 - schema inspection and hard inference/evaluation separation
for name,row in [("question",dev_raw[0]),("text",texts_raw[0]),("table",tables_raw[0]),("image",images_raw[0])]:
    print(name,{k:type(v).__name__ for k,v in row.items()})
@dataclass(frozen=True)
class QueryInput: qid:str; question:str
@dataclass(frozen=True)
class EvaluationOnly:
    gold_answers:List[str]; gold_support_ids:List[str]; gold_support_modalities:Dict[str,str]
    metadata_type:str="unknown"; gold_modalities:List[str]=field(default_factory=list)
    intermediate_answers:Any=None
FORBIDDEN={"answers","supporting_context","metadata","table_id","text_doc_ids","image_doc_ids",
           "modalities","type","intermediate_answers","gold_answers","gold_support_ids"}
def assert_safe_query(q:QueryInput):
    assert isinstance(q,QueryInput) and set(asdict(q))=={"qid","question"}
    assert not (set(asdict(q)) & FORBIDDEN)
def query_only(row:dict)->QueryInput:
    q=QueryInput(str(row.get("qid",row.get("id",""))),str(row["question"]))
    assert_safe_query(q); return q
def assert_inference_boundary(fn_name:str,q:QueryInput):
    assert_safe_query(q)
    assert set(asdict(q))=={"qid","question"},f"{fn_name} received a non-query field"
print("LEAKAGE CHECK PASSED")

In [ ]:
# Cell 7 - official image discovery, ZIP extraction and pixel validation (URLs never used)
def safe_extract(zp:Path,dst:Path):
    with zipfile.ZipFile(zp) as z:
        root=dst.resolve()
        for member in z.infolist():
            target=(dst/member.filename).resolve()
            if root not in target.parents and target!=root: raise ValueError(f"Unsafe ZIP member: {member.filename}")
        z.extractall(dst)
def discover_image_paths(records:List[dict])->Dict[str,Path]:
    expected={Path(str(r.get("path",""))).name for r in records if r.get("path")}
    mounts=Path("/kaggle/input"); extract=DIRS["images"]/"official"
    named=[] if not mounts.exists() else [p for p in mounts.rglob("final_dataset_images") if p.is_dir()]
    zips=[] if not mounts.exists() else [p for p in mounts.rglob("*.zip") if re.search(r"(mmqa|multimodal).*image|image.*(mmqa|multimodal)",p.name,re.I)]
    valid_zips=[]
    for z in zips:
        try:
            with zipfile.ZipFile(z) as zh:
                if any(Path(n).name in expected for n in zh.namelist()): valid_zips.append(z)
        except zipfile.BadZipFile: pass
    if not named and valid_zips:
        if len(valid_zips)>1: raise RuntimeError(f"Ambiguous official image ZIPs: {valid_zips}")
        marker=extract/".complete"
        if not marker.exists():
            extract.mkdir(parents=True,exist_ok=True); safe_extract(valid_zips[0],extract); marker.touch()
        named=[extract]
    roots=named or ([mounts] if mounts.exists() else [])
    by_name={}
    for root in roots:
        for p in root.rglob("*"):
            if p.is_file() and p.name in expected:
                if p.name in by_name and by_name[p.name]!=p: raise RuntimeError(f"Ambiguous image filename {p.name}")
                by_name[p.name]=p
    return {str(r["id"]):by_name[Path(str(r.get("path",""))).name] for r in records if Path(str(r.get("path",""))).name in by_name}
IMAGE_PATHS=discover_image_paths(images_raw)
ok=[]
for p in list(IMAGE_PATHS.values())[:5]:
    try:
        with Image.open(p) as im: im.verify()
        ok.append(p)
    except Exception as e: raise RuntimeError(f"Invalid official image {p}: {e}")
print(f"total image metadata records={len(images_raw):,}; resolved={len(IMAGE_PATHS):,}; missing={len(images_raw)-len(IMAGE_PATHS):,}; resolution={100*len(IMAGE_PATHS)/max(1,len(images_raw)):.2f}%")
if not IMAGE_PATHS: print("WARNING: official pixels unavailable; image retrieval evaluation is disabled. Text/table development only.")
else: print("PIL pixel checks passed:",len(ok))
FULL_IMAGE_BENCHMARK=bool(IMAGE_PATHS) and IMAGE_RETRIEVAL_MODE in {"hybrid","clip_full"}
if not RUN_RETRIEVAL_ONLY:
    assert len(images_raw)>0,"Official MMQA image metadata is required"
    assert len(IMAGE_PATHS)==len(images_raw),"End-to-end visual QA requires every official MMQA image to resolve"
    assert FULL_IMAGE_BENCHMARK and IMAGE_REASONING_USES_TITLE is False

In [ ]:
# Cell 8 - corpus normalization and table representations
def cell_text(c:Any)->str:
    if isinstance(c,dict): return str(c.get("text",c.get("value","")))
    return str(c)
def table_parts(row:dict)->Tuple[List[str],List[List[str]]]:
    t=row.get("table",row); header=t.get("header",[]); rows=t.get("table_rows",t.get("rows",[]))
    heads=[str(x.get("column_name",x.get("text",""))) if isinstance(x,dict) else str(x) for x in header]
    return heads,[[cell_text(c) for c in rr] for rr in rows]
def markdown_table(heads,rows)->str:
    width=max([len(heads),*(len(r) for r in rows)] or [0]); heads=(heads+[f"col_{i}" for i in range(len(heads),width)])[:width]
    clean=lambda s:str(s).replace("|","\\|").replace("\n"," ")
    return "\n".join(["| "+" | ".join(map(clean,heads))+" |","| "+" | ".join(["---"]*width)+" |",*["| "+" | ".join(map(clean,r+[""]*(width-len(r))))+" |" for r in rows]])
def chunk_table_rows(headers:List[str],rows:List[List[str]],max_rows=TABLE_CHUNK_ROWS,max_chars=6000):
    chunks=[]; start=0
    while start<len(rows):
        end=start; size=len(" | ".join(headers))
        while end<len(rows) and end-start<max_rows:
            row_size=sum(len(str(v)) for v in rows[end])+len(headers)*4
            if end>start and size+row_size>max_chars: break
            size+=row_size; end+=1
        chunks.append((start,max(start+1,end)))
        start=max(start+1,end)
    return chunks or [(0,0)]
def normalize_corpus():
    docs=[]
    for r in texts_raw:
        did=str(r["id"]); title=str(r.get("title","")); body=str(r.get("text",r.get("paragraph","")))
        docs.append({"doc_id":did,"chunk_id":did,"parent_doc_id":did,"parent_table_id":None,"modality":"text","title":title,"retrieval_text":f"{title}\n{body}","reasoning_content":body,"raw_content":body,"image_path":None,"row_start":None,"row_end":None,"headers":[]})
    for r in tables_raw:
        did=str(r["id"]); title=str(r.get("title",r.get("name",""))); h,rows=table_parts(r)
        spans=chunk_table_rows(h,rows)
        for start,end in spans:
            ch=rows[start:end]; cid=did if len(spans)==1 else f"{did}::rows_{start}-{max(start,end-1)}"
            linear=" ; ".join([f"{k}: {v}" for rr in ch for k,v in zip(h,rr)])
            neighbor_start=max(0,start-1); neighbor_end=min(len(rows),end+1)
            reasoning=markdown_table(h,rows[neighbor_start:neighbor_end])
            docs.append({"doc_id":cid,"chunk_id":cid,"parent_doc_id":did,"parent_table_id":did,"modality":"table","title":title,"retrieval_text":f"{title}\ncolumns: {' | '.join(h)}\n{linear}","reasoning_content":reasoning,"raw_content":reasoning,"image_path":None,"source_record":r,"row_start":start,"row_end":max(start,end-1),"headers":h})
    for r in images_raw:
        did=str(r["id"]); title=str(r.get("title",""))
        docs.append({"doc_id":did,"chunk_id":did,"parent_doc_id":did,"parent_table_id":None,"modality":"image","title":title,"retrieval_text":title,"reasoning_content":{"path":r.get("path")},"raw_content":{"path":r.get("path"),"url":r.get("url")},"image_path":str(IMAGE_PATHS[did]) if did in IMAGE_PATHS else None,"row_start":None,"row_end":None,"headers":[]})
    return docs
CORPUS=normalize_corpus(); BY_MODALITY={m:[d for d in CORPUS if d["modality"]==m] for m in ["text","table","image"]}
DOC_LOOKUP={(d["modality"],d["doc_id"]):d for d in CORPUS}
PARENT_LOOKUP=defaultdict(list)
for d in CORPUS:PARENT_LOOKUP[(d["modality"],d["parent_doc_id"])].append(d)
TABLE_PARENT_TO_CHUNKS={pid:chunks for (mod,pid),chunks in PARENT_LOOKUP.items() if mod=="table"}
print({m:len(v) for m,v in BY_MODALITY.items()},"table parents",len(TABLE_PARENT_TO_CHUNKS))
assert sum(len(v) for v in TABLE_PARENT_TO_CHUNKS.values())==len(BY_MODALITY["table"])
if not RUN_RETRIEVAL_ONLY:
    assert len(texts_raw)>200000 and len(tables_raw)>10000 and len(images_raw)>50000,"The full MMQA corpus was not loaded"
    assert len(BY_MODALITY["text"])==len(texts_raw) and len(TABLE_PARENT_TO_CHUNKS)==len(tables_raw) and len(BY_MODALITY["image"])==len(images_raw)
    print(f"FULL CORPUS CHECK PASSED: {len(texts_raw):,} text documents, {len(tables_raw):,} original tables, {len(images_raw):,} official images")

In [ ]:
# Cells 9-10 - cached normalized FAISS and BM25 indexes
def fingerprint(records):
    h=hashlib.sha256(); h.update(EMBED_MODEL.encode())
    for d in records: h.update((d["doc_id"]+"\0"+d["retrieval_text"]).encode("utf-8"))
    return h.hexdigest()
class DenseIndex:
    def __init__(self,modality,records,encoder): self.modality,self.records,self.encoder=modality,records,encoder; self.index=None
    def build_or_load(self):
        ip=DIRS["indexes"]/f"{self.modality}_faiss.index"; mp=DIRS["indexes"]/f"{self.modality}_doc_mapping.json"; ep=DIRS["cache"]/f"{self.modality}_embeddings.npy"
        fp=fingerprint(self.records)
        if ip.exists() and mp.exists() and ep.exists() and json.loads(mp.read_text())["fingerprint"]==fp: self.index=faiss.read_index(str(ip)); return
        emb=self.encoder.encode([d["retrieval_text"] for d in self.records],batch_size=128,normalize_embeddings=True,show_progress_bar=True).astype("float32")
        np.save(ep,emb); self.index=faiss.IndexFlatIP(emb.shape[1]); self.index.add(emb); faiss.write_index(self.index,str(ip))
        mp.write_text(json.dumps({"fingerprint":fp,"ids":[d["doc_id"] for d in self.records]}))
    def search(self,query,k,subquery="",query_source="original_question"):
        q=self.encoder.encode([query],normalize_embeddings=True).astype("float32"); scores,idx=self.index.search(q,min(k,len(self.records)))
        return [{**self.records[i],"dense_rank":rank,"dense_score":float(s),"bm25_rank":None,"bm25_score":None,"hybrid_rrf_score":0.0,"graph_rank":None,"graph_score":0.0,"final_score":float(s),"retrieved_by_subquery":[subquery or query],"query_provenance":[{"query_text":subquery or query,"query_source":query_source,"rank":rank,"score":float(s),"method":"dense"}]} for rank,(s,i) in enumerate(zip(scores[0],idx[0]),1) if i>=0]
def bm25_tokens(text:str)->List[str]:
    return re.findall(r"[a-z0-9]+",str(text).lower())
class BM25Index:
    def __init__(self,modality,records): self.modality,self.records=modality,records; self.index=None
    def build_or_load(self):
        path=DIRS["indexes"]/f"{self.modality}_bm25.pkl"; fp=fingerprint(self.records)
        if path.exists():
            payload=pickle.loads(path.read_bytes())
            if payload.get("fingerprint")==fp:self.index=payload["index"]; return
        self.index=BM25Okapi([bm25_tokens(d["retrieval_text"]) for d in self.records])
        path.write_bytes(pickle.dumps({"fingerprint":fp,"index":self.index},pickle.HIGHEST_PROTOCOL))
    def search(self,query,k,subquery="",query_source="original_question"):
        scores=np.asarray(self.index.get_scores(bm25_tokens(query))); k=min(k,len(scores))
        idx=np.argpartition(scores,-k)[-k:] if 0<k<len(scores) else np.arange(len(scores))
        idx=idx[np.argsort(scores[idx])[::-1]][:k]
        return [{**self.records[i],"dense_rank":None,"dense_score":None,"bm25_rank":rank,"bm25_score":float(scores[i]),"hybrid_rrf_score":0.0,"graph_rank":None,"graph_score":0.0,"final_score":float(scores[i]),"retrieved_by_subquery":[subquery or query],"query_provenance":[{"query_text":subquery or query,"query_source":query_source,"rank":rank,"score":float(scores[i]),"method":"bm25"}]} for rank,i in enumerate(idx,1)]
gpu_memory("before BGE:")
QUERY_ENCODER=SentenceTransformer(EMBED_MODEL,device="cuda:0" if torch.cuda.is_available() else "cpu")
TEXT_INDEX=DenseIndex("text",BY_MODALITY["text"],QUERY_ENCODER); TEXT_INDEX.build_or_load()
TABLE_INDEX=DenseIndex("table",BY_MODALITY["table"],QUERY_ENCODER); TABLE_INDEX.build_or_load()
IMAGE_TITLE_INDEX=DenseIndex("image_titles",BY_MODALITY["image"],QUERY_ENCODER); IMAGE_TITLE_INDEX.build_or_load()
QUERY_ENCODER.to("cpu")
for index in [TEXT_INDEX,TABLE_INDEX,IMAGE_TITLE_INDEX]:index.encoder=QUERY_ENCODER
gpu_memory("after BGE indexes / encoder moved to CPU:")
BM25_INDEXES={m:BM25Index(m,BY_MODALITY[m]) for m in ["text","table","image"]}
for index in BM25_INDEXES.values():index.build_or_load()
def rrf_fuse(dense:List[dict],lexical:List[dict],k:int)->List[dict]:
    merged={}
    for source,items in [("dense",dense),("bm25",lexical)]:
        for rank,d in enumerate(items,1):
            key=d["doc_id"]; x=merged.setdefault(key,{**d,"dense_rank":None,"dense_score":None,"bm25_rank":None,"bm25_score":None,"retrieved_by_subquery":[]})
            x[f"{source}_rank"]=rank; x[f"{source}_score"]=d.get(f"{source}_score")
            x["retrieved_by_subquery"]=list(dict.fromkeys(x["retrieved_by_subquery"]+d.get("retrieved_by_subquery",[]))); x["query_provenance"]=x.get("query_provenance",[])+[p for p in d.get("query_provenance",[]) if p not in x.get("query_provenance",[])]
    for x in merged.values():
        x["hybrid_rrf_score"]=(1/(RRF_K+x["dense_rank"]) if x["dense_rank"] else 0)+(1/(RRF_K+x["bm25_rank"]) if x["bm25_rank"] else 0)
        x["final_score"]=x["hybrid_rrf_score"]
    return sorted(merged.values(),key=lambda x:x["hybrid_rrf_score"],reverse=True)[:k]
def search_modality(modality:str,query:str,k:int,mode:str,subquery:str,query_source:str="original_question")->List[dict]:
    dense_index={"text":TEXT_INDEX,"table":TABLE_INDEX,"image":IMAGE_TITLE_INDEX}[modality]
    internal=max(k*3,50)
    if mode=="dense":return dense_index.search(query,k,subquery,query_source)
    if mode=="bm25":return BM25_INDEXES[modality].search(query,k,subquery,query_source)
    return rrf_fuse(dense_index.search(query,internal,subquery,query_source),BM25_INDEXES[modality].search(query,internal,subquery,query_source),k)

In [ ]:
# Cell 11 - official-image CLIP reranking with persistent lazy/full cache
class ClipReranker:
    def __init__(self): self.model=None; self.processor=None; self.cache={}; self.path=DIRS["cache"]/"clip_embeddings.pkl"
    @staticmethod
    def unwrap_features(output):
        # transformers versions differ: older CLIP returns Tensor; newer releases return BaseModelOutputWithPooling.
        if isinstance(output,torch.Tensor): return output
        pooled=getattr(output,"pooler_output",None)
        if isinstance(pooled,torch.Tensor): return pooled
        if isinstance(output,(tuple,list)):
            tensors=[x for x in output if isinstance(x,torch.Tensor) and x.ndim==2]
            if tensors:return tensors[-1]
        raise TypeError(f"Unsupported CLIP feature output: {type(output).__name__}")
    def load(self):
        from transformers import CLIPModel,CLIPProcessor
        self.processor=CLIPProcessor.from_pretrained(CLIP_MODEL,use_fast=False); self.model=CLIPModel.from_pretrained(CLIP_MODEL).eval().to(CLIP_DEVICE)
        if self.path.exists(): self.cache=pickle.loads(self.path.read_bytes())
    def encode_missing(self,docs):
        if self.model is None: self.load()
        missing=[d for d in docs if d["image_path"] and d["doc_id"] not in self.cache]
        for start in range(0,len(missing),32):
            batch=missing[start:start+32]; ims=[]
            for d in batch:
                with Image.open(d["image_path"]) as im:
                    if im.mode=="P" and "transparency" in im.info: im=im.convert("RGBA")
                    ims.append(im.convert("RGB"))
            x=self.processor(images=ims,return_tensors="pt").to(self.model.device)
            with torch.inference_mode(): z=self.unwrap_features(self.model.get_image_features(**x)); z=z/z.norm(dim=-1,keepdim=True)
            self.cache.update({d["doc_id"]:e.cpu().numpy().astype("float32") for d,e in zip(batch,z)})
        if missing: self.path.write_bytes(pickle.dumps(self.cache,pickle.HIGHEST_PROTOCOL))
    def rerank(self,query,docs,k):
        docs=[d for d in docs if d.get("image_path")]; self.encode_missing(docs)
        if not docs:return []
        text_limit=int(getattr(self.model.config.text_config,"max_position_embeddings",77))
        x=self.processor(text=[query],return_tensors="pt",padding=True,truncation=True,max_length=text_limit).to(self.model.device)
        assert x["input_ids"].shape[-1]<=text_limit
        with torch.inference_mode(): q=self.unwrap_features(self.model.get_text_features(**x)); q=q/q.norm(dim=-1,keepdim=True)
        q=q[0].cpu().numpy(); out=[]
        for d in docs:
            score=float(np.dot(q,self.cache[d["doc_id"]])); out.append({**d,"clip_score":score,"final_score":score})
        return sorted(out,key=lambda x:x["final_score"],reverse=True)[:k]
CLIP=ClipReranker()
def retrieve_images(query,k=IMAGE_TOP_K):
    if not IMAGE_PATHS:return []
    pool=IMAGE_TITLE_INDEX.search(query,len(BY_MODALITY["image"]) if IMAGE_RETRIEVAL_MODE=="clip_full" else IMAGE_CANDIDATES,query)
    pool=[d for d in pool if d.get("image_path")]
    if IMAGE_RETRIEVAL_MODE=="metadata_only": return pool[:k]
    return CLIP.rerank(query,pool,k)

In [ ]:
# Cell 12 - original-table graph and bounded cross-modal expansion
def norm_entity(x): return re.sub(r"\s+"," ",str(x).strip().lower())
def cell_entities(cell:Any)->List[str]:
    values=[cell_text(cell)]
    if isinstance(cell,dict):
        for key in ["links","entities","wiki_links"]:
            for link in cell.get(key,[]) or []:
                if isinstance(link,dict):values.extend([link.get("title",""),link.get("text","")])
                else:values.append(link)
    return list(dict.fromkeys("e::"+norm_entity(v) for v in values if 1<len(norm_entity(v))<=100 and not norm_entity(v).replace(".","").isdigit()))
def build_graph():
    path=DIRS["cache"]/"mmqa_graph_v2.pkl"; fp="v2-"+fingerprint(CORPUS)
    if path.exists():
        payload=pickle.loads(path.read_bytes())
        if payload["fingerprint"]==fp:return payload["graph"]
    g=nx.Graph()
    for modality,raws in [("text",texts_raw),("image",images_raw)]:
        for raw in raws:
            did=str(raw["id"]); dn=f"d::{modality}::{did}"; title=norm_entity(raw.get("title",""))
            g.add_node(dn,kind="document",doc_id=did,modality=modality)
            if title:g.add_edge(dn,"e::"+title,kind="title")
    for raw in tables_raw:
        did=str(raw["id"]); dn=f"d::table::{did}"; title=norm_entity(raw.get("title",raw.get("name","")))
        g.add_node(dn,kind="document",doc_id=did,modality="table")
        if title:g.add_edge(dn,"e::"+title,kind="title")
        table=raw.get("table",raw); rows=table.get("table_rows",table.get("rows",[]))
        for row in rows:
            entities=list(dict.fromkeys(e for cell in row for e in cell_entities(cell)))[:20]
            for entity in entities:g.add_edge(dn,entity,kind="cell")
            for i,a in enumerate(entities):
                for b in entities[i+1:i+4]:g.add_edge(a,b,kind="row")
    path.write_bytes(pickle.dumps({"fingerprint":fp,"graph":g},pickle.HIGHEST_PROTOCOL)); return g
GRAPH=build_graph() if ENABLE_GRAPH else None
class WebRetriever:
    """Extension point only; web evidence is forbidden in official MMQA experiments."""
    def search(self,query:str,k:int=5):
        if not ENABLE_WEB: raise RuntimeError("Web retrieval is disabled for reproducibility and leakage control")
        raise NotImplementedError("Provide a versioned, auditable backend before enabling non-benchmark experiments")
def representative_record(modality:str,parent_id:str)->Optional[dict]:
    records=PARENT_LOOKUP.get((modality,parent_id),[])
    return records[0] if records else None
def cross_modal_graph_expand(seed_candidates:Dict[str,List[dict]])->Tuple[List[dict],List[dict]]:
    if not GRAPH:return [],[]
    seeds=sorted([d for xs in seed_candidates.values() for d in xs],key=lambda x:x.get("final_score",0),reverse=True)[:GRAPH_SEED_K]
    found={}; provenance=[]; seed_ids={(d["modality"],d["parent_doc_id"]) for d in seeds}
    for seed_rank,seed in enumerate(seeds,1):
        start=f'd::{seed["modality"]}::{seed["parent_doc_id"]}'
        if start not in GRAPH:continue
        queue=[(start,0,[start])]; visited={start}
        while queue:
            node,hop,path=queue.pop(0)
            if hop>=2:continue
            neighbors=sorted(GRAPH.neighbors(node),key=str)[:GRAPH_MAX_DEGREE]
            for nxt in neighbors:
                if nxt in visited:continue
                visited.add(nxt); new_path=path+[nxt]; new_hop=hop+1
                attrs=GRAPH.nodes[nxt]
                if attrs.get("kind")=="document":
                    key=(attrs["modality"],attrs["doc_id"])
                    if key not in seed_ids:
                        score=1/((RRF_K+seed_rank)*(new_hop+1)); old=found.get(key)
                        rec=representative_record(*key)
                        if rec and (old is None or score>old["graph_score"]):
                            found[key]={**rec,"dense_rank":None,"dense_score":None,"bm25_rank":None,"bm25_score":None,"hybrid_rrf_score":0.0,"graph_score":score,"final_score":score,"retrieved_by_subquery":seed.get("retrieved_by_subquery",[])}
                        if len(provenance)<GRAPH_MAX_EXPANDED_DOCS*4:
                            provenance.append({"seed_doc":seed["parent_doc_id"],"seed_modality":seed["modality"],"entity_path":new_path[1:-1],"expanded_doc":attrs["doc_id"],"expanded_modality":attrs["modality"],"hop_count":new_hop,"graph_score":score})
                queue.append((nxt,new_hop,new_path))
    ranked=sorted(found.values(),key=lambda x:x["graph_score"],reverse=True)[:GRAPH_MAX_EXPANDED_DOCS]
    for rank,d in enumerate(ranked,1):d["graph_rank"]=rank
    return ranked,provenance

In [ ]:
# Cell 13 - deterministic question-only planner for retrieval-only experiments
def question_only_plan(question:str)->dict:
    q=question.lower(); confidence={"text":0.50,"table":0.45,"image":0.20}
    if re.search(r"\b(color|colour|logo|pictured|shown|image|photo|looks?|wearing|flag|shape)\b",q):confidence["image"]=0.88
    if re.search(r"\b(how many|number|year|when|date|rank|score|record|role|list|first|last|most|least)\b",q):confidence["table"]=0.78
    if re.search(r"\b(according|described|stated|why|who|biography|history|where)\b",q):confidence["text"]=0.72
    relational=bool(re.search(r"\b(where|which|whose|that|who)\b.*\b(where|which|what|when|color|colour|year)\b",q))
    if relational:
        confidence["table"]=max(confidence["table"],0.70); confidence["text"]=max(confidence["text"],0.60)
    available=["text","table"]+(["image"] if IMAGE_PATHS else [])
    selected=[m for m in available if confidence[m]>=PLANNER_MODALITY_THRESHOLD]
    if not selected:selected=[max(available,key=lambda m:confidence[m])]
    routing_confidence=max(confidence[m] for m in selected)
    return {"needs_decomposition":False,"subqueries":[question],"needed_modalities":selected,"modality_confidence":confidence,"routing_confidence":routing_confidence,"needs_relational_search":relational,"planner_backend":"question_only_rules"}
def fallback_plan(question):return question_only_plan(question)

In [ ]:
# Cell 14 - retrieval memory checkpoint
if CLIP.model is not None: CLIP.model.to("cpu")
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
gpu_memory("retrieval stack ready:")

In [ ]:
# Cell 15 - one explicit, reproducible, memory-bounded Qwen load
QWEN_CALL_COUNTER=defaultdict(int); model=None; processor=None
if RUN_RETRIEVAL_ONLY:
    print("RUN_RETRIEVAL_ONLY=True: Qwen is NOT loaded")
else:
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
    from qwen_vl_utils import process_vision_info
    gpu_memory("before Qwen:")
    processor=AutoProcessor.from_pretrained(MODEL_NAME,min_pixels=256*28*28,max_pixels=384*28*28,use_fast=IMAGE_PROCESSOR_USE_FAST)
    bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
    max_memory={i:"10GiB" for i in range(torch.cuda.device_count())}; max_memory["cpu"]="24GiB"
    model=Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,quantization_config=bnb,device_map="balanced",max_memory=max_memory,
        offload_folder=str(DIRS["offload"]),offload_state_dict=True,low_cpu_mem_usage=True,
        attn_implementation="sdpa",
    ).eval()
    assert model is not None and processor is not None
    print("QWEN LOADED")
    print("IMAGE PROCESSOR use_fast =",IMAGE_PROCESSOR_USE_FAST)
    print("MAMMQA GENERATION SETTINGS:")
    print("do_sample =",MAMMQA_DO_SAMPLE); print("temperature =",MAMMQA_TEMPERATURE)
    print("top_p =",MAMMQA_TOP_P); print("max_new_tokens =",MAX_LLM_NEW_TOKENS)
    print("stage_max_new_tokens =",STAGE_MAX_NEW_TOKENS); print("format_repair_max_new_tokens =",FORMAT_REPAIR_NEW_TOKENS)
    gpu_memory("GPU memory status:")

In [ ]:
# Cell 16 - protected-prompt token budgeting, structured generation and one format-repair retry
def append_jsonl_log(path,row):
    with Path(path).open("a",encoding="utf-8") as f:f.write(json.dumps(row,ensure_ascii=False,default=str)+"\n"); f.flush()

def parse_json_object(text):
    text=str(text or ""); candidates=[text]
    candidates += re.findall(r"```(?:json)?\s*(\{.*?\})\s*```",text,re.S|re.I)
    start=text.find("{"); end=text.rfind("}")
    if start>=0 and end>start:candidates.append(text[start:end+1])
    for candidate in candidates:
        try:
            parsed=json.loads(candidate)
            if isinstance(parsed,dict):return parsed
        except Exception:pass
    return None

def clip01(x,default=0.0):
    try:return max(0.0,min(1.0,float(x)))
    except:return default

def stable_call_seed(qid,stage,retry_count):
    return (SEED+int(hashlib.sha256(f"{qid}:{stage}:{retry_count}".encode()).hexdigest()[:8],16))%(2**31-1)

def public_evidence(item):
    return {k:v for k,v in item.items() if k not in {"image_path","rank","score","retrieval_method","query_provenance"}}

def evidence_id(item):return str(item.get("evidence_id",item.get("doc_id","unknown")))

def build_messages(system,protected,evidence):
    content=[]
    for item in evidence:
        if item.get("modality")=="image" and item.get("image_path") and Path(item["image_path"]).is_file():
            content.append({"type":"image","image":f'file://{item["image_path"]}'})
    content.append({"type":"text","text":json.dumps({"protected":protected,"evidence":[public_evidence(x) for x in evidence]},ensure_ascii=False)})
    return [{"role":"system","content":system},{"role":"user","content":content}]

def processor_inputs(messages):
    prompt=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    images,videos=process_vision_info(messages)
    kwargs={"text":[prompt],"return_tensors":"pt","padding":True,"truncation":False}
    if images:kwargs.update(images=images,videos=videos)
    inputs=processor(**kwargs)
    return inputs,int(inputs["input_ids"].shape[-1]),len(images or [])

def shorten_lowest_ranked_text_evidence(evidence):
    order=sorted(range(len(evidence)),key=lambda i:(evidence[i].get("rank",0),i),reverse=True)
    for index in order:
        content=evidence[index].get("content")
        if isinstance(content,str) and len(content)>600:
            new=copy.deepcopy(evidence); new[index]["content"]=content[:max(600,len(content)//2)]+"\n[deterministically shortened]"
            return new,True
    return evidence,False

def prepare_bounded_inputs(system,protected,evidence,max_budget=MAX_INPUT_TOKENS):
    work=copy.deepcopy(evidence); removed=[]; trimmed=False
    while True:
        messages=build_messages(system,protected,work); inputs,count,image_count=processor_inputs(messages)
        if count<=max_budget:
            assert count<=max_budget
            return inputs,messages,count,image_count,work,removed,trimmed
        del inputs; gc.collect()
        shortened,did_shorten=shorten_lowest_ranked_text_evidence(work)
        if did_shorten:
            work=shortened; trimmed=True; continue
        if work:
            index=max(range(len(work)),key=lambda i:(work[i].get("rank",0),i))
            removed.append(evidence_id(work[index])); work=work[:index]+work[index+1:]; trimmed=True; continue
        # Protected prompt alone must never be truncated.
        raise RuntimeError(f"PROTECTED_PROMPT_TOO_LARGE: {count}>{max_budget}")

def validate_schema(stage,obj):
    if not isinstance(obj,dict):return False
    if stage=="planner":return isinstance(obj.get("subqueries",[]),list) and isinstance(obj.get("modality_confidence",{}),dict)
    if stage.startswith("stage1_"):return all(k in obj for k in ["modality","evidence_summary","useful_subquestions","candidate_answer","confidence","evidence_ids"])
    if stage.startswith("stage2_"):return all(k in obj for k in ["insights","subanswers","candidate_answer","confidence","insufficient","supporting_doc_ids"])
    if stage=="verifier":return obj.get("status") in {"supported","insufficient","conflict"}
    if stage=="aggregator":return obj.get("status") in {"answered","abstain"} and bool(str(obj.get("answer","")).strip())
    return True

def stage_output_budget(stage,requested):
    if stage.endswith("_format_repair"):return min(requested,FORMAT_REPAIR_NEW_TOKENS)
    key="stage1" if stage.startswith("stage1_") else ("stage2" if stage.startswith("stage2_") else stage)
    return min(requested,STAGE_MAX_NEW_TOKENS.get(key,requested))

def generate_once(system,protected,evidence,qid,stage,do_sample,max_new_tokens,retry_count=0):
    max_new_tokens=stage_output_budget(stage,max_new_tokens)
    inputs,messages,input_count,image_count,used,removed,trimmed=prepare_bounded_inputs(system,protected,evidence)
    seed=stable_call_seed(qid,stage,retry_count); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    inputs=inputs.to(model.device); started=time.perf_counter()
    print(f"[QWEN START] qid={qid} stage={stage} input_tokens={input_count} images={image_count} max_new_tokens={max_new_tokens}",flush=True)
    generate_kwargs={"max_new_tokens":max_new_tokens,"do_sample":do_sample}
    if do_sample:generate_kwargs.update(temperature=MAMMQA_TEMPERATURE,top_p=MAMMQA_TOP_P)
    with torch.inference_mode():ids=model.generate(**inputs,**generate_kwargs)
    output_ids=ids[:,inputs.input_ids.shape[1]:]
    raw=processor.batch_decode(output_ids,skip_special_tokens=True)[0].strip()
    meta={"qid":qid,"stage":stage,"input_token_count":input_count,"max_input_budget":MAX_INPUT_TOKENS,
          "image_count":image_count,"output_token_count":int(output_ids.shape[-1]),"max_new_tokens":max_new_tokens,
          "retry_count":retry_count,"generation_time":time.perf_counter()-started,"context_trimmed":trimmed,
          "removed_evidence_ids":removed,"protected_prompt_truncated":False,"seed":seed}
    print(f"[QWEN DONE] qid={qid} stage={stage} output_tokens={meta['output_token_count']} seconds={meta['generation_time']:.1f} parse_pending=True",flush=True)
    del inputs,ids,output_ids; gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return raw,meta

REPAIR_PROMPT="""Reformat the previous response into the required JSON schema. Do not add facts, evidence, reasoning, or answers that were not present in the previous response. Return only valid JSON."""

def structured_call(system,protected,evidence,qid,stage,do_sample,max_new_tokens=MAX_LLM_NEW_TOKENS):
    raw,meta=generate_once(system,protected,evidence,qid,stage,do_sample,max_new_tokens,0)
    parsed=parse_json_object(raw); ok=validate_schema(stage,parsed); meta["parse_success"]=ok
    print(f"[PARSE] qid={qid} stage={stage} success={ok}",flush=True)
    append_jsonl_log(GENERATION_DIAGNOSTICS_PATH,meta)
    if meta["context_trimmed"]:append_jsonl_log(TRUNCATION_DIAGNOSTICS_PATH,meta)
    repair_raw=None; diagnostics=[meta]
    if not ok:
        repair_protected={"required_stage":stage,"previous_response":raw[:6000],"instruction":"Formatting repair only; add no new facts."}
        repair_raw,rmeta=generate_once(REPAIR_PROMPT,repair_protected,[],qid,stage+"_format_repair",False,FORMAT_REPAIR_NEW_TOKENS,1)
        repaired=parse_json_object(repair_raw); ok=validate_schema(stage,repaired); rmeta["parse_success"]=ok
        append_jsonl_log(GENERATION_DIAGNOSTICS_PATH,rmeta); diagnostics.append(rmeta)
        print(f"[REPAIR PARSE] qid={qid} stage={stage} success={ok}",flush=True)
        if ok:parsed=repaired
    return {"raw_output":raw,"repair_raw_output":repair_raw,"parsed_output":parsed if ok else None,
            "parse_success":bool(ok),"retry_count":int(repair_raw is not None),"generation_schema_failure":not ok,
            "generation_diagnostics":diagnostics}

def compact_parsed(envelope):
    if envelope and envelope.get("parse_success"):return envelope["parsed_output"]
    return {"schema_failure":True,"raw_output":str((envelope or {}).get("raw_output",""))[:3000]}

def canonical_answer(value):
    answer=str(value or "").strip().strip('"\'')
    low=answer.lower().strip(" .")
    if low in {"yes","yeah","true"}:return "yes"
    if low in {"no","false"}:return "no"
    if re.fullmatch(r"[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?",answer):return answer.replace(",","")
    return answer

def parse_final_answer(envelope):
    parsed=(envelope or {}).get("parsed_output") if (envelope or {}).get("parse_success") else None
    if isinstance(parsed,dict):
        for key in ["answer","final_answer","best_answer","candidate_answer"]:
            if str(parsed.get(key,"")).strip():return canonical_answer(parsed[key]),True
    raw=str((envelope or {}).get("raw_output","") or "")
    tags=re.findall(r"<answer>\s*(.*?)\s*</answer>",raw,re.S|re.I)
    if tags:return canonical_answer(tags[-1]),True
    labels=re.findall(r"(?:final\s+answer|answer)\s*[:\-]\s*(.+)",raw,re.I)
    if labels:return canonical_answer(labels[-1]),True
    plain=raw.strip().splitlines()[0].strip() if raw.strip() else ""
    if plain and len(plain)<=120 and not plain.startswith(("{","[","<")):return canonical_answer(plain),True
    return "",False

In [ ]:
# Cell 17 - robust question-only planner with original-question preservation
PLANNER_PROMPT="""Plan full-corpus retrieval using only the question. Return JSON with needs_decomposition, subqueries (maximum 3; strings or objects with query text), needed_modalities, modality_confidence with text/table/image probabilities, routing_confidence, and needs_relational_search. Do not use hidden metadata or gold evidence."""

def normalize_subquery(value):
    if isinstance(value,str):return value.strip()
    if isinstance(value,dict):
        for key in ["query","subquery","question","text"]:
            if isinstance(value.get(key),str) and value[key].strip():return value[key].strip()
    return ""

def assert_clean_retrieval_query(query):
    stripped=str(query).lstrip()
    assert not stripped.startswith(("{'query':",'{"query":'))
    assert not re.search(r"['\"]modality_confidence['\"]\s*:",str(query))

def build_query_records(question,subqueries):
    records=[]; seen=set(); original=str(question).strip()
    for value,source in [(original,"original_question")]+[(x,"decomposed_subquery") for x in subqueries]:
        text=original if source=="original_question" else normalize_subquery(value)
        key=" ".join(text.split()).casefold()
        if text and key not in seen:
            assert_clean_retrieval_query(text); records.append({"text":text,"source":source}); seen.add(key)
    assert records and records[0]["text"]==original and records[0]["source"]=="original_question"
    return records

def fallback_plan(question,reason="planner_disabled"):
    base=question_only_plan(question); subs=[]; records=build_query_records(question,subs)
    return {**base,"raw_planner_output":None,"parsed_planner_output":None,"planner_parse_success":True,
            "planner_schema_failure":False,"normalized_subqueries":subs,"retrieval_queries":records,"fallback_reason":reason}

def plan_query(q):
    assert_safe_query(q)
    envelope=structured_call(PLANNER_PROMPT,{"qid":q.qid,"question":q.question},[],q.qid,"planner",PLANNER_DO_SAMPLE)
    parsed=envelope.get("parsed_output") or {}; raw_subs=parsed.get("subqueries",[])
    normalized=[]
    for value in raw_subs:
        text=normalize_subquery(value)
        if text:normalized.append(text)
    records=build_query_records(q.question,normalized)
    mods=[m for m in parsed.get("needed_modalities",[]) if m in {"text","table","image"}]
    conf={m:clip01(parsed.get("modality_confidence",{}).get(m),0.5) for m in ["text","table","image"]}
    if not IMAGE_PATHS:mods=[m for m in mods if m!="image"]
    if not mods:mods=["text","table"]+(["image"] if IMAGE_PATHS else [])
    plan={"needs_decomposition":bool(parsed.get("needs_decomposition",len(normalized)>0)),"subqueries":[r["text"] for r in records],
          "normalized_subqueries":normalized,"retrieval_queries":records,"needed_modalities":mods,"modality_confidence":conf,
          "routing_confidence":clip01(parsed.get("routing_confidence"),max(conf[m] for m in mods)),
          "needs_relational_search":bool(parsed.get("needs_relational_search",False)),"planner_backend":"qwen",
          "raw_planner_output":envelope["raw_output"],"parsed_planner_output":parsed if envelope["parse_success"] else None,
          "planner_parse_success":envelope["parse_success"],"planner_schema_failure":envelope["generation_schema_failure"],
          "planner_generation":envelope}
    assert all(r["source"] in {"original_question","decomposed_subquery"} for r in records)
    return plan

In [ ]:
# Cell 18 - soft-priority routing, full candidate retrieval and separate context selection
def ensure_encoder():
    assert QUERY_ENCODER is not None,"Persistent query encoder was not initialized"; return QUERY_ENCODER

def route_budgets(plan,router_mode):
    assert router_mode in {"hard","soft","all"}
    available=["text","table"]+(["image"] if IMAGE_PATHS else [])
    normal={"text":TEXT_CONTEXT_K,"table":TABLE_CONTEXT_K,"image":IMAGE_CONTEXT_K}
    fallback={"text":FALLBACK_TEXT_K,"table":FALLBACK_TABLE_K,"image":FALLBACK_IMAGE_K}
    selected=set(plan.get("needed_modalities",[]))&set(available)
    if not selected:selected={max(available,key=lambda m:plan.get("modality_confidence",{}).get(m,0))}
    low_conf=float(plan.get("routing_confidence",0))<ROUTER_CONFIDENCE_THRESHOLD
    # Soft routing is prioritization only: every modality is searched at the full candidate depth.
    if router_mode in {"soft","all"}:
        candidates={m:max(RETRIEVAL_CANDIDATE_K,MIN_METRIC_CANDIDATE_K) for m in available}
        if router_mode=="all" or low_conf:contexts={m:normal[m] for m in available}
        else:contexts={m:(normal[m] if m in selected else fallback[m]) for m in available}
        return candidates,contexts
    return ({m:max(RETRIEVAL_CANDIDATE_K,MIN_METRIC_CANDIDATE_K) for m in selected},{m:normal[m] for m in selected})

def merge_subquery_rankings(groups,k):
    merged={}
    for group in groups:
        for rank,d in enumerate(group,1):
            key=d["doc_id"]; x=merged.setdefault(key,{**d,"subquery_rrf_score":0.0,"retrieved_by_subquery":[],"query_provenance":[]})
            x["subquery_rrf_score"]+=1/(RRF_K+rank)
            x["retrieved_by_subquery"]=list(dict.fromkeys(x["retrieved_by_subquery"]+d.get("retrieved_by_subquery",[])))
            x["query_provenance"]+= [p for p in d.get("query_provenance",[]) if p not in x["query_provenance"]]
            for field in ["dense_score","bm25_score","hybrid_rrf_score"]:
                values=[v for v in [x.get(field),d.get(field)] if v is not None]; x[field]=max(values) if values else None
    for x in merged.values():x["final_score"]=x["subquery_rrf_score"]
    return sorted(merged.values(),key=lambda x:x["final_score"],reverse=True)[:k]

def aggregate_parent_candidates(items,k):
    groups=defaultdict(list)
    for d in items:groups[d["parent_doc_id"]].append(d)
    parents=[]
    for parent_id,chunks in groups.items():
        chunks=sorted(chunks,key=lambda x:x.get("final_score",0),reverse=True); best=dict(chunks[0])
        best["matched_chunks"]=[{"chunk_id":x["doc_id"],"row_start":x.get("row_start"),"row_end":x.get("row_end"),"score":x.get("final_score",0)} for x in chunks[:5]]
        best["retrieved_by_subquery"]=list(dict.fromkeys(q for x in chunks for q in x.get("retrieved_by_subquery",[])))
        best["query_provenance"]=[p for x in chunks for p in x.get("query_provenance",[])]
        parents.append(best)
    return sorted(parents,key=lambda x:x.get("final_score",0),reverse=True)[:k]

def fuse_graph(base,expanded,k_by_mod):
    out={m:list(base.get(m,[])) for m in ["text","table","image"]}; graph_by_mod=defaultdict(list)
    for d in expanded:graph_by_mod[d["modality"]].append(d)
    for mod in out:
        base_rank={d["parent_doc_id"]:i+1 for i,d in enumerate(out[mod])}; graph_rank={d["parent_doc_id"]:i+1 for i,d in enumerate(graph_by_mod[mod])}
        records={d["parent_doc_id"]:dict(d) for d in out[mod]}
        for d in graph_by_mod[mod]:records.setdefault(d["parent_doc_id"],dict(d))
        fused=[]
        for did,d in records.items():
            d["graph_rank"]=graph_rank.get(did); d["graph_score"]=next((x.get("graph_score",0) for x in graph_by_mod[mod] if x["parent_doc_id"]==did),0)
            d["final_score"]=(1/(RRF_K+base_rank[did]) if did in base_rank else 0)+(1/(RRF_K+graph_rank[did]) if did in graph_rank else 0)
            if d["graph_rank"]:d.setdefault("query_provenance",[]).append({"query_text":"GRAPH_EXPANSION","query_source":"graph_expansion","rank":d["graph_rank"],"score":d["graph_score"],"method":"graph"})
            fused.append(d)
        out[mod]=sorted(fused,key=lambda x:x["final_score"],reverse=True)[:k_by_mod.get(mod,RETRIEVAL_CANDIDATE_K)]
    return out

def select_context(question,candidates,context_budgets):
    context={m:[] for m in ["text","table","image"]}
    for mod,k in context_budgets.items():
        pool=candidates.get(mod,[])
        context[mod]=CLIP.rerank(question,[d for d in pool if d.get("image_path")],k) if mod=="image" and pool else pool[:k]
    return context

def retrieve_evidence(q,plan,retrieval_mode=RETRIEVAL_MODE,router_mode=ROUTER_MODE,use_graph=True,fixed_all=False):
    assert_safe_query(q); ensure_encoder(); started=time.perf_counter()
    if fixed_all:router_mode="all"
    candidate_budgets,context_budgets=route_budgets(plan,router_mode)
    query_records=plan.get("retrieval_queries") or build_query_records(q.question,plan.get("normalized_subqueries",[]))
    assert query_records[0]=={"text":q.question,"source":"original_question"}
    candidates={m:[] for m in ["text","table","image"]}
    for mod,k in candidate_budgets.items():
        groups=[search_modality(mod,qr["text"],max(k,RETRIEVAL_CANDIDATE_K),retrieval_mode,qr["text"],qr["source"]) for qr in query_records]
        chunks=merge_subquery_rankings(groups,max(k*3,RETRIEVAL_CANDIDATE_K)); candidates[mod]=aggregate_parent_candidates(chunks,k)
        assert len(candidates[mod])>=min(MIN_METRIC_CANDIDATE_K,len(BY_MODALITY[mod]))
    graph_provenance=[]
    if use_graph:
        expanded,graph_provenance=cross_modal_graph_expand(candidates)
        limits={m:max(candidate_budgets.get(m,0),MIN_METRIC_CANDIDATE_K) for m in candidates}
        candidates=fuse_graph(candidates,expanded,limits)
    context=select_context(q.question,candidates,context_budgets)
    priorities=sorted(({"modality":m,"confidence":float(plan.get("modality_confidence",{}).get(m,0)),"candidate_budget":candidate_budgets[m],"context_budget":context_budgets.get(m,0)} for m in candidate_budgets),key=lambda x:(-x["confidence"],x["modality"]))
    diagnostic={"qid":q.qid,"question":q.question,"raw_planner_output":plan.get("raw_planner_output"),
                "parsed_planner_output":plan.get("parsed_planner_output"),"normalized_subqueries":plan.get("normalized_subqueries",[]),
                "retrieval_queries":query_records,"modality_confidence":plan.get("modality_confidence",{}),
                "modalities_searched":list(candidate_budgets),"candidate_counts":{m:len(xs) for m,xs in candidates.items()},
                "context_counts":{m:len(xs) for m,xs in context.items()}}
    append_jsonl_log(PLANNER_DIAGNOSTICS_PATH,diagnostic)
    return {"retrieval_candidates":candidates,"context_evidence":context,"graph_provenance":graph_provenance,
            "candidate_ids":{m:[d["parent_doc_id"] for d in xs] for m,xs in candidates.items()},
            "context_ids":{m:[d["parent_doc_id"] for d in xs] for m,xs in context.items()},
            "routing":{"mode":router_mode,"retrieval_mode":retrieval_mode,"candidate_budgets":candidate_budgets,"context_budgets":context_budgets,"searched_modalities":list(candidate_budgets),"soft_router_priorities":priorities},
            "retrieval_latency_seconds":time.perf_counter()-started}

In [ ]:
# Cell 19 - modality experts over clean retrieved evidence only
EXPERT_PROMPT="""Use only the supplied evidence for the named modality. Extract relevant facts, propose useful subquestions, and indicate uncertainty. Return only JSON with exactly these fields: modality, evidence_summary, useful_subquestions, candidate_answer, confidence, evidence_ids. candidate_answer must be INSUFFICIENT when the evidence does not support an answer. confidence is 0..1. evidence_ids must be copied from supplied evidence. No outside knowledge or hidden metadata."""

def clean_evidence(items,modality,content_chars=2400):
    clean=[]
    for rank,d in enumerate(items,1):
        body=d.get("reasoning_content",d.get("raw_content",""))
        if modality=="image":body="Official MultiModalQA image pixels are attached."
        elif not isinstance(body,str):body=json.dumps(body,ensure_ascii=False,default=str)
        clean.append({"evidence_id":str(d["parent_doc_id"]),"modality":modality,
                      "content":body[:content_chars],"image_path":d.get("image_path") if modality=="image" else None,
                      "rank":rank})
    return clean

def run_modality_experts(q,evidence):
    assert_inference_boundary("run_modality_experts",q); out={}
    for mod in ["text","table","image"]:
        items=[d for d in evidence.get(mod,[]) if mod!="image" or d.get("image_path")]
        if not items:continue
        ev=clean_evidence(items,mod); allowed={x["evidence_id"] for x in ev}
        protected={"qid":q.qid,"question":q.question,"modality":mod,
                   "required_schema":{"modality":mod,"evidence_summary":"string","useful_subquestions":["short strings"],"candidate_answer":"string or INSUFFICIENT","confidence":"0..1","evidence_ids":["supplied ids"]}}
        envelope=structured_call(EXPERT_PROMPT,protected,ev,q.qid,f"stage1_{mod}",MAMMQA_DO_SAMPLE)
        parsed=envelope.get("parsed_output")
        if parsed:
            parsed["modality"]=mod; parsed["confidence"]=clip01(parsed.get("confidence"))
            parsed["evidence_ids"]=[x for x in map(str,parsed.get("evidence_ids",[])) if x in allowed]
            if not str(parsed.get("candidate_answer","")).strip():parsed["candidate_answer"]="INSUFFICIENT"
        out[mod]=envelope
    return out

In [ ]:
# Cell 20 - anchor-specific Stage II: anchor insight plus raw non-anchor evidence
SYNTH_PROMPT="""Synthesize an answer anchored on the supplied anchor expert. The evidence array contains only raw evidence from other modalities. Return only JSON with exactly these fields: insights, subanswers, candidate_answer, confidence, insufficient, supporting_doc_ids. Use no outside knowledge and do not provide chain-of-thought. supporting_doc_ids must be copied from supplied records."""

def run_cross_modal_agents(q,experts,evidence):
    assert_inference_boundary("run_cross_modal_agents",q); out={}
    allowed={str(d["parent_doc_id"]) for xs in evidence.values() for d in xs}
    for anchor,expert_envelope in experts.items():
        other=[]
        for mod in ["text","table","image"]:
            if mod!=anchor:other.extend(clean_evidence(evidence.get(mod,[]),mod,1800))
        protected={"qid":q.qid,"question":q.question,"anchor_modality":anchor,
                   "anchor_expert":compact_parsed(expert_envelope),
                   "required_schema":{"insights":["short strings"],"subanswers":["short strings"],"candidate_answer":"string or INSUFFICIENT","confidence":"0..1","insufficient":"boolean","supporting_doc_ids":["supplied ids"]}}
        envelope=structured_call(SYNTH_PROMPT,protected,other,q.qid,f"stage2_{anchor}",MAMMQA_DO_SAMPLE)
        parsed=envelope.get("parsed_output")
        if parsed:
            parsed["confidence"]=clip01(parsed.get("confidence"))
            parsed["supporting_doc_ids"]=[x for x in map(str,parsed.get("supporting_doc_ids",[])) if x in allowed]
            parsed["insufficient"]=bool(parsed.get("insufficient",False))
        out[anchor]=envelope
    return out

In [ ]:
# Cell 21 - verifier: infrastructure errors never become epistemic insufficiency
VERIFY_PROMPT="""Verify the supplied candidate syntheses against the evidence. Return only JSON with exactly these fields: status, best_answer, confidence, supporting_doc_ids, contradicting_doc_ids, missing_information, retry_modality, retry_query. status must be supported, insufficient, or conflict. retry_modality must be text, table, image, or none. Use only supplied evidence."""

def verify_answer(q,synth,evidence):
    assert_inference_boundary("verify_answer",q)
    allowed={str(d["parent_doc_id"]) for xs in evidence.values() for d in xs}
    ev=[]
    for mod in ["text","table","image"]:ev.extend(clean_evidence(evidence.get(mod,[]),mod,1600))
    protected={"qid":q.qid,"question":q.question,"candidate_syntheses":{m:compact_parsed(x) for m,x in synth.items()},
               "required_status_values":["supported","insufficient","conflict"],"required_retry_modalities":["text","table","image","none"]}
    envelope=structured_call(VERIFY_PROMPT,protected,ev,q.qid,"verifier",VERIFIER_DO_SAMPLE)
    parsed=envelope.get("parsed_output")
    if parsed:
        parsed["confidence"]=clip01(parsed.get("confidence"))
        for key in ["supporting_doc_ids","contradicting_doc_ids"]:
            parsed[key]=[x for x in map(str,parsed.get(key,[])) if x in allowed]
        if parsed.get("retry_modality") not in {"text","table","image","none"}:parsed["retry_modality"]="none"
    envelope["effective_status"]=parsed.get("status") if parsed else "verifier_error"
    envelope["verifier_error"]=not bool(parsed)
    return envelope

In [ ]:
# Cell 22 - bounded feedback, shared reasoning stages and answer aggregation
AGG_PROMPT="""Choose the answer supported by the supplied syntheses. Return only JSON with exactly these fields: answer, confidence, status, supporting_doc_ids. status must be answered or abstain. If evidence is insufficient, return answer ABSTAIN and status abstain. No explanation and no outside knowledge."""

def run_reasoning(q,evidence):
    experts=run_modality_experts(q,evidence)
    synth=run_cross_modal_agents(q,experts,evidence)
    return experts,synth

def aggregate_answer(q,synth,sees_question=True):
    assert_inference_boundary("aggregate_answer",q)
    protected={"qid":q.qid,"syntheses":{m:compact_parsed(x) for m,x in synth.items()}}
    if sees_question:protected["question"]=q.question
    return structured_call(AGG_PROMPT,protected,[],q.qid,"aggregator",MAMMQA_DO_SAMPLE)

def merge_evidence(old,new):
    out={}
    for mod in ["text","table","image"]:
        vals={d["parent_doc_id"]:d for d in old.get(mod,[])}
        for d in new.get(mod,[]):vals.setdefault(d["parent_doc_id"],d)
        out[mod]=list(vals.values())
    return out

def refresh_bundle_ids(bundle):
    bundle["candidate_ids"]={m:[d["parent_doc_id"] for d in xs] for m,xs in bundle["retrieval_candidates"].items()}
    bundle["context_ids"]={m:[d["parent_doc_id"] for d in xs] for m,xs in bundle["context_evidence"].items()}

def run_pipeline(q,cfg):
    assert_inference_boundary("run_pipeline",q); started=time.perf_counter()
    plan=fallback_plan(q.question) if not cfg["planner"] else plan_query(q)
    bundle=retrieve_evidence(q,plan,retrieval_mode=cfg.get("retrieval_mode",RETRIEVAL_MODE),router_mode=cfg.get("router_mode",ROUTER_MODE),use_graph=cfg["graph"],fixed_all=not cfg["router"])
    evidence=bundle["context_evidence"]; experts,synth=run_reasoning(q,evidence)
    verification=verify_answer(q,synth,evidence) if cfg["verifier"] else None
    rounds=1; added=[]; targeted=None
    retrieval_round_details=[{"round":1,"query_records":plan["retrieval_queries"],"bundle":copy.deepcopy(bundle)}]
    reasoning_round_details=[{"round":1,"experts":experts,"cross_modal":synth,"verification":verification}]
    verifier_parsed=(verification or {}).get("parsed_output") if (verification or {}).get("parse_success") else None
    if cfg["feedback"] and verifier_parsed and verifier_parsed.get("status") in {"insufficient","conflict"}:
        mod=verifier_parsed.get("retry_modality"); rq=str(verifier_parsed.get("retry_query","")).strip()
        if mod in {"text","table","image"} and rq and rounds<MAX_RETRIEVAL_ROUNDS:
            assert_clean_retrieval_query(rq)
            retry_plan={"retrieval_queries":build_query_records(q.question,[rq]),"normalized_subqueries":[rq],"needed_modalities":[mod],
                        "modality_confidence":{m:float(m==mod) for m in ["text","table","image"]},"routing_confidence":1.0}
            retry=retrieve_evidence(q,retry_plan,retrieval_mode=cfg.get("retrieval_mode",RETRIEVAL_MODE),router_mode="hard",use_graph=cfg["graph"])
            before={d["parent_doc_id"] for xs in evidence.values() for d in xs}
            evidence=merge_evidence(evidence,retry["context_evidence"])
            bundle["context_evidence"]=evidence; bundle["retrieval_candidates"]=merge_evidence(bundle["retrieval_candidates"],retry["retrieval_candidates"])
            bundle["graph_provenance"]+=retry["graph_provenance"]; refresh_bundle_ids(bundle)
            after={d["parent_doc_id"] for xs in evidence.values() for d in xs}; added=sorted(after-before); rounds=2
            targeted={"retry_triggered":True,"retry_reason":verifier_parsed["status"],"retry_modality":mod,"retry_query":rq,
                      "new_candidate_ids":retry["candidate_ids"],"new_context_ids":retry["context_ids"],"gold_evidence_newly_recovered":None}
            retrieval_round_details.append({"round":2,"query_records":retry_plan["retrieval_queries"],"bundle":copy.deepcopy(retry)})
            experts,synth=run_reasoning(q,evidence); verification=verify_answer(q,synth,evidence)
            reasoning_round_details.append({"round":2,"experts":experts,"cross_modal":synth,"verification":verification})
    agg=aggregate_answer(q,synth,cfg["agg_question"]); answer,answer_ok=parse_final_answer(agg)
    agg_parsed=agg.get("parsed_output") if agg.get("parse_success") else None
    final_status=agg_parsed.get("status") if agg_parsed else "generation_error"
    final_ver=(verification or {}).get("parsed_output") if (verification or {}).get("parse_success") else None
    forced=False
    if cfg["verifier"] and final_ver and final_ver.get("status") in {"insufficient","conflict"}:
        answer="ABSTAIN"; final_status="abstain"; forced=True; answer_ok=True
    elif agg_parsed and agg_parsed.get("status")=="abstain":
        answer="ABSTAIN"; final_status="abstain"; answer_ok=True
    final={"answer":answer,"status":final_status,"confidence":clip01((agg_parsed or {}).get("confidence")),
           "supporting_doc_ids":(agg_parsed or {}).get("supporting_doc_ids",[]),"forced_by_valid_verifier":forced,
           "answer_parse_success":answer_ok,"infrastructure_failure":not answer_ok}
    return {"qid":q.qid,"question":q.question,"inference_input_fields":["qid","question"],"planner":plan,**bundle,"retrieved":evidence,
            "stage1":experts,"cross_modal":synth,"verification":verification,"aggregator_raw":agg,"raw_final_output":agg.get("raw_output"),
            "parsed_final_answer":agg.get("parsed_output"),"final_decision":final,"retrieval_round_details":retrieval_round_details,
            "reasoning_round_details":reasoning_round_details,"targeted_feedback":targeted,"retrieval_rounds":rounds,"new_docs_added":added,
            "final_answer":answer,"confidence":final["confidence"],"abstain":answer=="ABSTAIN","forced_parser_abstain":False,
            "latency_seconds":time.perf_counter()-started}

In [ ]:
# Cell 23 - exact controls; MAMMQA_ORACLE is a pure reasoning upper bound
CONFIGS={
 "B1_Dense":{"planner":False,"router":False,"router_mode":"all","retrieval_mode":"dense","graph":False,"verifier":False,"feedback":False,"agg_question":True},
 "B2_Planner_Dense":{"planner":True,"router":True,"router_mode":"soft","retrieval_mode":"dense","graph":False,"verifier":False,"feedback":False,"agg_question":True},
 "FULL":{"planner":True,"router":True,"router_mode":"soft","retrieval_mode":"hybrid","graph":True,"verifier":True,"feedback":True,"agg_question":True},
}

def oracle_evidence(ev):
    evidence={m:[] for m in ["text","table","image"]}
    for did in ev.gold_support_ids:
        mod=ev.gold_support_modalities.get(did)
        recs=PARENT_LOOKUP.get((mod,did),[]) if mod else []
        for rec in recs:
            if rec and (rec["modality"]!="image" or rec.get("image_path")):
                evidence[rec["modality"]].append({**rec,"dense_score":1.0,"bm25_score":1.0,"final_score":1.0,"query_provenance":[{"query_text":"ORACLE_GOLD_EVIDENCE","query_source":"oracle","rank":1,"score":1.0,"method":"oracle"}]})
    return evidence

def oracle_pipeline(q,ev):
    assert_inference_boundary("oracle_pipeline",q); started=time.perf_counter(); evidence=oracle_evidence(ev)
    experts,synth=run_reasoning(q,evidence); agg=aggregate_answer(q,synth,True); answer,ok=parse_final_answer(agg)
    parsed=agg.get("parsed_output") if agg.get("parse_success") else None
    if parsed and parsed.get("status")=="abstain":answer="ABSTAIN"; ok=True
    final={"answer":answer,"status":parsed.get("status") if parsed else "generation_error","confidence":clip01((parsed or {}).get("confidence")),"answer_parse_success":ok,"infrastructure_failure":not ok}
    return {"qid":q.qid,"question":q.question,"inference_input_fields":["qid","question"],"planner":{"oracle":True},
            "retrieval_candidates":{},"context_evidence":evidence,"retrieved":evidence,"candidate_ids":{},"context_ids":{m:[d["parent_doc_id"] for d in xs] for m,xs in evidence.items()},
            "stage1":experts,"cross_modal":synth,"verification":None,"aggregator_raw":agg,"raw_final_output":agg.get("raw_output"),
            "parsed_final_answer":agg.get("parsed_output"),"final_decision":final,"retrieval_round_details":[],
            "reasoning_round_details":[{"round":1,"experts":experts,"cross_modal":synth,"verification":None}],"targeted_feedback":None,
            "retrieval_rounds":0,"new_docs_added":[],"final_answer":answer,"confidence":final["confidence"],"abstain":answer=="ABSTAIN",
            "forced_parser_abstain":False,"latency_seconds":time.perf_counter()-started}

In [ ]:
# Cell 24 - gold parsing and candidate/context retrieval metrics
def answer_strings(row):
    vals=[]
    def walk(x):
        if isinstance(x,str):vals.append(x)
        elif isinstance(x,list):
            for y in x:walk(y)
        elif isinstance(x,dict):
            for k in ["answer","answers","aliases","normalized_answer"]:
                if k in x:walk(x[k])
    walk(row.get("answers",[])); return list(dict.fromkeys(v.strip() for v in vals if v.strip()))

def support_info(row):
    ids=[]; mods={}; known={m:{str(x["id"]) for x in raw} for m,raw in [("text",texts_raw),("table",tables_raw),("image",images_raw)]}
    def add(v,mod=None):
        s=str(v); inferred=mod or next((m for m,z in known.items() if s in z),None)
        if s not in {"","None"} and inferred and s in known[inferred]:ids.append(s); mods[s]=inferred
    def walk(x,mod=None):
        if isinstance(x,list):
            for y in x:walk(y,mod)
        elif isinstance(x,dict):
            local={"text":"text","table":"table","image":"image"}.get(str(x.get("modality",x.get("doc_part",mod))).lower(),mod)
            for k,v in x.items():
                km={"text_doc_id":"text","table_id":"table","image_doc_id":"image","doc_id":local}.get(k)
                if km:add(v,km)
                elif k not in {"answer","text","content"}:walk(v,local)
        elif mod:add(x,mod)
    walk(row.get("supporting_context",[])); return list(dict.fromkeys(ids)),mods

def evaluation_only(row):
    ids,mods=support_info(row); meta=row.get("metadata",{})
    return EvaluationOnly(answer_strings(row),ids,mods,str(meta.get("type","unknown")),list(meta.get("modalities",[])),row.get("intermediate_answers"))

ARTICLES={"a","an","the"}
def norm_answer(s):
    s=str(s).lower(); s="".join(" " if c in string.punctuation else c for c in s); return " ".join(w for w in s.split() if w not in ARTICLES)
def em_f1(pred,golds):
    p=norm_answer(pred).split(); best_em=0; best_f1=0.0
    for g0 in golds:
        g=norm_answer(g0).split(); best_em=max(best_em,int(p==g)); common=Counter(p)&Counter(g); n=sum(common.values())
        score=0 if not p or not g or not n else 2*(n/len(p))*(n/len(g))/((n/len(p))+(n/len(g)))
        best_f1=max(best_f1,score)
    return best_em,best_f1

def ranked_parent_ids(result,k,mod=None,pool="retrieval_candidates"):
    pools=result.get(pool,{}) or {}; docs=pools.get(mod,[]) if mod else [x for xs in pools.values() for x in xs]
    docs=sorted(docs,key=lambda x:x.get("final_score",0),reverse=True)
    return list(dict.fromkeys(str(x["parent_doc_id"]) for x in docs))[:k]

def retrieval_scores(result,ev):
    out={}; gold=set(map(str,ev.gold_support_ids))
    for k in [1,3,5,10]:
        ids=ranked_parent_ids(result,k); hit=len(gold&set(ids)); out[f"recall@{k}"]=hit/len(gold) if gold else math.nan
        out[f"precision@{k}"]=hit/len(ids) if ids else 0.0; out[f"hit@{k}"]=int(hit>0) if gold else math.nan
        out[f"all_support_recall@{k}"]=int(gold<=set(ids)) if gold else math.nan
        for mod in ["text","table","image"]:
            gm={x for x in gold if ev.gold_support_modalities.get(x)==mod}; mids=set(ranked_parent_ids(result,k,mod))
            out[f"{mod}_recall@{k}"]=len(gm&mids)/len(gm) if gm else math.nan
    ranked=ranked_parent_ids(result,10**9); out["mrr"]=next((1/(i+1) for i,x in enumerate(ranked) if x in gold),0.0) if gold else math.nan
    candidate=set(ranked); context=set(ranked_parent_ids(result,10**9,pool="context_evidence"))
    out["gold_candidate_pool_coverage"]=len(gold&candidate)/len(gold) if gold else math.nan
    out["gold_context_coverage"]=len(gold&context)/len(gold) if gold else math.nan
    out["gold_in_candidate_pool"]=int(gold<=candidate) if gold else math.nan; out["gold_in_context"]=int(gold<=context) if gold else math.nan
    out["retrieved_document_count"]=len(candidate); out["context_document_count"]=len(context)
    gold_mods=set(ev.gold_support_modalities.values()); selected=set(result.get("planner",{}).get("needed_modalities",[]))
    out["planner_modality_recall"]=len(gold_mods&selected)/len(gold_mods) if gold_mods else math.nan
    return out

In [ ]:
# Cell 25 - frozen failed-run sample: exact qids and exact order
FROZEN_QIDS=[
 "f7dd1ac143507bd2ab9a480f18b86706","254afe636a34c3a5d94efa22d95465ad",
 "ebb69554283456251305b4b8ab579f76","4b9a4868455f98a69a8494ef1dfe7205",
 "186146ea1f5a9f7ada7f7fd31c5c75a4","c9fcae0ac6e3aa3f08375db9e4fe9037",
 "5dfd4afc175cdebf43295316a540d70c","6d372c501004c3e448b1991f39160e99",
 "b5d7685e595423638d26dd60e99fcc39","3126664c6ce707c6aca2c475ace8db08",
 "183a1eec0e38eaee513a5ec7996f55f1","2a5fdab4235e28037389777f473c4f9f",
 "2f154508615dc2d34b4d4b7fdced0b0e","22e22f3b558eeae2178ddd9cb6bc2b04",
 "1e3f1a08d865b3676a5ca81d8adc270b","ceb2b6a63d5f3b5eb8075979a55e7471",
]
row_by_qid={query_only(r).qid:r for r in dev_raw}
missing=[qid for qid in FROZEN_QIDS if qid not in row_by_qid]
assert not missing,f"Frozen qids missing from the loaded dev split: {missing}"
SELECTED=[row_by_qid[qid] for qid in FROZEN_QIDS]; SELECTED_QIDS=[query_only(r).qid for r in SELECTED]
assert SELECTED_QIDS==FROZEN_QIDS and len(set(SELECTED_QIDS))==16
failed_selected=FAILED_OUTPUT_ROOT/"selected_questions.json"
if failed_selected.exists():
    old=json.loads(failed_selected.read_text(encoding="utf-8")); old_qids=[str(x.get("qid",x.get("id",""))) for x in old]
    assert SELECTED_QIDS==old_qids,"Frozen repair qids differ from failed selected_questions.json"
IMAGE_SELECTED_QIDS={query_only(r).qid for r in SELECTED if "image" in set(evaluation_only(r).gold_support_modalities.values())}
(OUTPUT_ROOT/"selected_questions.json").write_text(json.dumps([asdict(query_only(r)) for r in SELECTED],indent=2,ensure_ascii=False),encoding="utf-8")
print("Frozen qids verified:",len(SELECTED_QIDS)); print("selected qids =",SELECTED_QIDS)

In [ ]:
# Cell 26 - static/preflight invariants (no answer-quality tuning)
assert build_query_records("Original question?",[{"query":"focused query"},"second query"])[0]=={"text":"Original question?","source":"original_question"}
assert normalize_subquery({"query":"table lookup"})=="table lookup"
for bad in ["{'query': 'bad'}",'{"query": "bad"}']:
    try:assert_clean_retrieval_query(bad)
    except AssertionError:pass
    else:raise AssertionError("Serialized planner dictionary was accepted as a retrieval query")
test_plan={"needed_modalities":["text"],"modality_confidence":{"text":.9,"table":.1,"image":.1},"routing_confidence":.9}
candidate_budget,context_budget=route_budgets(test_plan,"soft")
available=[m for m in ["text","table","image"] if BY_MODALITY[m] and (m!="image" or IMAGE_PATHS)]
assert set(candidate_budget)==set(available)
assert all(candidate_budget[m]>=MIN_METRIC_CANDIDATE_K for m in available)
assert context_budget["text"]>=context_budget.get("table",0)
assert parse_json_object('{"status":"supported"}')["status"]=="supported"
assert validate_schema("verifier",{"status":"supported"})
assert not validate_schema("verifier",{})
assert MAX_INPUT_TOKENS==4096 and FORMAT_REPAIR_NEW_TOKENS<MAX_LLM_NEW_TOKENS
print("PREFLIGHT PASSED: frozen sample, clean planner queries, soft candidate coverage, schemas, and token budgets")

In [ ]:
# Cell 27 - isolated resumable rerun of exactly four requested methods
def append_jsonl(path,row):
    with Path(path).open("a",encoding="utf-8") as f:f.write(json.dumps(row,ensure_ascii=False,default=str)+"\n"); f.flush(); os.fsync(f.fileno())

def read_jsonl_checked(path):
    path=Path(path)
    if not path.exists():return []
    rows=[]
    with path.open(encoding="utf-8") as f:
        for no,line in enumerate(f,1):
            if line.strip():
                try:rows.append(json.loads(line))
                except json.JSONDecodeError as e:raise ValueError(f"{path}:{no}: {e}") from e
    return rows

def completed(path):
    rows=read_jsonl_checked(path); qids=[r["qid"] for r in rows]
    assert len(qids)==len(set(qids)),f"Duplicate qids in {path}"
    assert qids==FROZEN_QIDS[:len(qids)],f"Stale, foreign, or out-of-order qids in {path}"
    return set(qids)

def clean_for_json(x):
    if isinstance(x,Path):return str(x)
    if isinstance(x,dict):return {k:clean_for_json(v) for k,v in x.items() if k!="source_record"}
    if isinstance(x,(list,tuple)):return [clean_for_json(v) for v in x]
    if isinstance(x,np.generic):return x.item()
    return x

def annotate_pool_ranks(result):
    z=clean_for_json(result)
    for key in ["retrieval_candidates","context_evidence","retrieved"]:
        for mod,docs in (z.get(key,{}) or {}).items():
            for rank,d in enumerate(docs,1):d.setdefault("candidate_rank",rank)
    return z

EXPERIMENT_CONFIG={
 "experiment_id":EXPERIMENT_ID,"failed_experiment_id":FAILED_EXPERIMENT_ID,"seed":SEED,"model":MODEL_NAME,
 "embed_model":EMBED_MODEL,"clip_model":CLIP_MODEL,"run_mode":RUN_MODE,"run_retrieval_only":RUN_RETRIEVAL_ONLY,
 "retrieval_candidate_k":RETRIEVAL_CANDIDATE_K,"min_metric_candidate_k":MIN_METRIC_CANDIDATE_K,
 "context_k":{"text":TEXT_CONTEXT_K,"table":TABLE_CONTEXT_K,"image":IMAGE_CONTEXT_K},
 "fallback_context_k":{"text":FALLBACK_TEXT_K,"table":FALLBACK_TABLE_K,"image":FALLBACK_IMAGE_K},
 "max_input_tokens":MAX_INPUT_TOKENS,"max_new_tokens":MAX_LLM_NEW_TOKENS,"stage_max_new_tokens":STAGE_MAX_NEW_TOKENS,"format_repair_new_tokens":FORMAT_REPAIR_NEW_TOKENS,
 "sampling":{"mam_mqa":{"do_sample":MAMMQA_DO_SAMPLE,"temperature":MAMMQA_TEMPERATURE,"top_p":MAMMQA_TOP_P},"planner":False,"verifier":False},
 "max_retrieval_rounds":MAX_RETRIEVAL_ROUNDS,"enable_web":ENABLE_WEB,"question_qids":SELECTED_QIDS,
 "methods":{"MAMMQA_ORACLE":{"oracle_gold_evidence":True,"verifier":False},**CONFIGS},
}
config_path=DIRS["logs"]/"config.json"
if config_path.exists():assert json.loads(config_path.read_text(encoding="utf-8"))==EXPERIMENT_CONFIG,"Experiment id exists with a different config"
else:config_path.write_text(json.dumps(EXPERIMENT_CONFIG,indent=2),encoding="utf-8")

def run_method(name,rows):
    path=DIRS["predictions"]/f"{name}.jsonl"; done=completed(path)
    print(f"\n[METHOD START] {name}: {len(done)}/16 already complete",flush=True)
    method_started=time.perf_counter()
    for position,raw in enumerate(rows,1):
        q=query_only(raw)
        if q.qid in done:
            print(f"[{name}] {position}/16 RESUME-SKIP qid={q.qid}",flush=True); continue
        question_started=time.perf_counter()
        print(f"[{name}] {position}/16 START qid={q.qid}",flush=True)
        assert_inference_boundary(name,q)
        if name=="MAMMQA_ORACLE":
            ev=evaluation_only(raw); result=oracle_pipeline(q,ev)
        else:
            result=run_pipeline(q,CONFIGS[name])
            # Gold is materialized only after the query-only pipeline has produced its final answer.
            ev=evaluation_only(raw)
        em,f1=em_f1(result["final_answer"],ev.gold_answers); result=annotate_pool_ranks(result)
        metrics={} if name=="MAMMQA_ORACLE" else retrieval_scores(result,ev)
        result.update(gold_answers=ev.gold_answers,gold_support_ids=ev.gold_support_ids,gold_support_modalities=ev.gold_support_modalities,
                      metadata_type=ev.metadata_type,gold_modalities=ev.gold_modalities,exact_match=em,token_f1=f1,retrieval_metrics=metrics)
        if result.get("targeted_feedback"):
            new_ids={str(x) for xs in result["targeted_feedback"]["new_candidate_ids"].values() for x in xs}
            result["targeted_feedback"]["gold_evidence_newly_recovered"]=bool(set(map(str,ev.gold_support_ids))&new_ids)
        append_jsonl(path,result); done.add(q.qid)
        print(f"[{name}] {position}/16 SAVED qid={q.qid} answer={result['final_answer']!r} seconds={time.perf_counter()-question_started:.1f}",flush=True)
    print(f"[METHOD DONE] {name}: 16/16 seconds={time.perf_counter()-method_started:.1f}",flush=True)
    return path

METHODS=["MAMMQA_ORACLE","B1_Dense","B2_Planner_Dense","FULL"]
assert METHODS==["MAMMQA_ORACLE","B1_Dense","B2_Planner_Dense","FULL"] and RUN_MODE=="smoke" and RUN_RETRIEVAL_ONLY is False
assert model is not None and processor is not None,"Qwen did not load"
RESULT_PATHS={name:run_method(name,SELECTED) for name in METHODS}
ROWS_BY_METHOD={name:read_jsonl_checked(path) for name,path in RESULT_PATHS.items()}
for name,rows in ROWS_BY_METHOD.items():
    assert [r["qid"] for r in rows]==FROZEN_QIDS,f"{name} failed the exact-qid/order requirement"
    assert len(rows)==16 and all(r.get("inference_input_fields")==["qid","question"] for r in rows)
for name in ["B1_Dense","B2_Planner_Dense","FULL"]:
    for r in ROWS_BY_METHOD[name]:
        available=[m for m in ["text","table","image"] if BY_MODALITY[m] and (m!="image" or IMAGE_PATHS)]
        assert set(r["routing"]["searched_modalities"])==set(available)
        assert all(len(r["retrieval_candidates"].get(m,[]))>=min(MIN_METRIC_CANDIDATE_K,len(BY_MODALITY[m])) for m in available)
        records=r["planner"]["retrieval_queries"]
        assert records[0]=={"text":r["question"],"source":"original_question"}
        assert all(not str(x["text"]).lstrip().startswith(("{'query':",'{"query":')) for x in records)
assert all(r.get("verification") is None for r in ROWS_BY_METHOD["MAMMQA_ORACLE"]),"Oracle is not pure"
print("RERUN COMPLETE: exact four methods x exact 16 qids")

In [ ]:
# Cell 28 - pipeline health, requested reports, failure analysis, traces and before/after audit
def mean_valid(vals):
    vals=[float(v) for v in vals if v is not None and not (isinstance(v,float) and math.isnan(v))]
    return float(np.mean(vals)) if vals else math.nan

def envelopes_for(row,kind):
    if kind=="planner":return [row.get("planner",{}).get("planner_generation")]
    if kind=="stage1":return [x for rr in row.get("reasoning_round_details",[]) for x in rr.get("experts",{}).values()]
    if kind=="stage2":return [x for rr in row.get("reasoning_round_details",[]) for x in rr.get("cross_modal",{}).values()]
    if kind=="verifier":return [rr.get("verification") for rr in row.get("reasoning_round_details",[]) if rr.get("verification") is not None]
    if kind=="aggregator":return [row.get("aggregator_raw")]
    return []

def parse_rate(rows,kind):
    env=[x for r in rows for x in envelopes_for(r,kind) if x]
    return sum(bool(x.get("parse_success")) for x in env)/len(env) if env else math.nan

def stage_health(rows,kind):
    env=[x for r in rows for x in envelopes_for(r,kind) if x]
    generated=sum(bool(str(x.get("raw_output","")).strip()) for x in env)
    parsed=sum(bool(x.get("parse_success")) for x in env)
    return {"generated":generated,"parsed":parsed,"total":len(env)}

def all_parent_ids(row,pool):return {str(d["parent_doc_id"]) for xs in (row.get(pool,{}) or {}).values() for d in xs}
def answer_matches(value,golds):return em_f1(str(value or ""),golds)[0]==1
def intermediate_answers(row):
    vals=[]
    for rr in row.get("reasoning_round_details",[]):
        for env in list(rr.get("experts",{}).values())+list(rr.get("cross_modal",{}).values()):
            parsed=(env or {}).get("parsed_output") or {}; vals.append(parsed.get("candidate_answer",""))
        ver=(rr.get("verification") or {}).get("parsed_output") or {}; vals.append(ver.get("best_answer",""))
    return [x for x in vals if str(x).strip()]

def classify_failure(row):
    if row["exact_match"]:return "correct"
    planner=envelopes_for(row,"planner")
    if any(x and x.get("generation_schema_failure") for x in planner):return "planner_query_parse_failure"
    queries=row.get("planner",{}).get("retrieval_queries",[])
    if not queries or queries[0].get("text")!=row["question"]:return "planner_failure"
    if any(str(x.get("text","")).lstrip().startswith(("{'query':",'{"query":')) for x in queries):return "planner_query_parse_failure"
    if set(row.get("routing",{}).get("searched_modalities",[]))!={m for m in ["text","table","image"] if BY_MODALITY[m] and (m!="image" or IMAGE_PATHS)}:return "routing_failure"
    if any(x.get("generation_schema_failure") for x in envelopes_for(row,"stage1") if x):return "stage1_generation_failure"
    if any(x.get("generation_schema_failure") for x in envelopes_for(row,"stage2") if x):return "stage2_generation_failure"
    if any(x.get("generation_schema_failure") for x in envelopes_for(row,"verifier") if x):return "verifier_generation_failure"
    if any(x.get("generation_schema_failure") for x in envelopes_for(row,"aggregator") if x):return "aggregation_generation_failure"
    if not row.get("final_decision",{}).get("answer_parse_success"):return "answer_parsing_failure"
    gold=set(map(str,row.get("gold_support_ids",[]))); candidate=all_parent_ids(row,"retrieval_candidates"); context=all_parent_ids(row,"context_evidence")
    if gold and not gold<=candidate:return "retrieval_failure"
    if gold and not gold<=context:return "context_selection_failure"
    earlier_correct=any(answer_matches(v,row.get("gold_answers",[])) for v in intermediate_answers(row))
    if earlier_correct and row.get("verification") and ((row["verification"].get("parsed_output") or {}).get("status")!="supported"):return "verification_failure"
    if earlier_correct:return "aggregation_failure"
    if row.get("retrieval_rounds")==2 and row.get("targeted_feedback") and not row["targeted_feedback"].get("gold_evidence_newly_recovered"):return "feedback_retrieval_failure"
    image_gold={x for x in gold if row.get("gold_support_modalities",{}).get(x)=="image"}
    if image_gold and image_gold<=context:return "vision_failure"
    stage1_answers=[]
    for env in envelopes_for(row,"stage1"):stage1_answers.append(((env or {}).get("parsed_output") or {}).get("candidate_answer",""))
    if not any(answer_matches(x,row.get("gold_answers",[])) for x in stage1_answers):return "expert_extraction_failure"
    stage2_answers=[]
    for env in envelopes_for(row,"stage2"):stage2_answers.append(((env or {}).get("parsed_output") or {}).get("candidate_answer",""))
    if not any(answer_matches(x,row.get("gold_answers",[])) for x in stage2_answers):return "cross_modal_synthesis_failure"
    if row.get("abstain"):return "abstention_failure"
    return "aggregation_failure"

def feedback_recovery(row):
    gold=set(map(str,row.get("gold_support_ids",[]))); added=set(map(str,row.get("new_docs_added",[])))
    first=[]; rounds=row.get("reasoning_round_details",[])
    if rounds:
        for env in list(rounds[0].get("experts",{}).values())+list(rounds[0].get("cross_modal",{}).values()):first.append(((env or {}).get("parsed_output") or {}).get("candidate_answer",""))
    return bool(gold&added),bool(row.get("retrieval_rounds")==2 and row.get("exact_match") and not any(answer_matches(x,row.get("gold_answers",[])) for x in first))

summary=[]; retrieval_summary=[]
for name in METHODS:
    rows=ROWS_BY_METHOD[name]; oracle=name=="MAMMQA_ORACLE"
    summary.append({"Method":name,"n":len(rows),"R@5":math.nan if oracle else mean_valid([r["retrieval_metrics"].get("recall@5") for r in rows]),
                    "R@10":math.nan if oracle else mean_valid([r["retrieval_metrics"].get("recall@10") for r in rows]),
                    "EM":mean_valid([r["exact_match"] for r in rows]),"F1":mean_valid([r["token_f1"] for r in rows]),
                    "Abstention":mean_valid([r["abstain"] for r in rows]),"correct_count":sum(r["exact_match"] for r in rows)})
    if not oracle:
        retrieval_summary.append({"Method":name,**{f"Recall@{k}":mean_valid([r["retrieval_metrics"].get(f"recall@{k}") for r in rows]) for k in [1,3,5,10]},
                                  "MRR":mean_valid([r["retrieval_metrics"].get("mrr") for r in rows]),
                                  "candidate_gold_coverage":mean_valid([r["retrieval_metrics"].get("gold_candidate_pool_coverage") for r in rows]),
                                  "context_gold_coverage":mean_valid([r["retrieval_metrics"].get("gold_context_coverage") for r in rows])})
SDF=pd.DataFrame(summary); RDF=pd.DataFrame(retrieval_summary); FULL_ROWS=ROWS_BY_METHOD["FULL"]

perq=[]; failures=[]
for i,r in enumerate(FULL_ROWS,1):
    kind=classify_failure(r); recovered,corrected=feedback_recovery(r); rm=r["retrieval_metrics"]
    perq.append({"#":i,"qid":r["qid"],"short_question":r["question"][:90],"gold_answer":" | ".join(r["gold_answers"]),"prediction":r["final_answer"],
                 "EM":r["exact_match"],"F1":r["token_f1"],"Recall@5":rm.get("recall@5"),"gold_found_R@5":bool(rm.get("hit@5",0)),
                 "candidate_gold_found":bool(rm.get("gold_in_candidate_pool",0)),"context_gold_found":bool(rm.get("gold_in_context",0)),
                 "planner_modalities":"+".join(r.get("planner",{}).get("needed_modalities",[])),
                 "feedback_used":r.get("retrieval_rounds")==2,"abstained":r["abstain"],"failure_type":kind})
    failures.append({"qid":r["qid"],"failure_type":kind,"candidate_gold_coverage":rm.get("gold_candidate_pool_coverage"),
                     "context_gold_coverage":rm.get("gold_context_coverage"),"feedback_recovered_evidence":recovered,"feedback_corrected_answer":corrected})
PQDF=pd.DataFrame(perq); FDF=pd.DataFrame(failures)

generation_rows=read_jsonl_checked(GENERATION_DIAGNOSTICS_PATH)
non_oracle=ROWS_BY_METHOD["B1_Dense"]+ROWS_BY_METHOD["B2_Planner_Dense"]+FULL_ROWS
all_rows=sum(list(ROWS_BY_METHOD.values()),[])
health={
 "questions_completed":{name:len(ROWS_BY_METHOD[name]) for name in METHODS},
 "planner":stage_health(ROWS_BY_METHOD["B2_Planner_Dense"]+FULL_ROWS,"planner"),
 "stage1":stage_health(all_rows,"stage1"),"stage2":stage_health(all_rows,"stage2"),
 "verifier":stage_health(FULL_ROWS,"verifier"),"aggregator":stage_health(all_rows,"aggregator"),
 "mean_input_tokens":mean_valid([x.get("input_token_count") for x in generation_rows]),"mean_output_tokens":mean_valid([x.get("output_token_count") for x in generation_rows]),
 "format_repair_calls":sum(x.get("retry_count",0)>0 or str(x.get("stage","")).endswith("_format_repair") for x in generation_rows),
 "context_trim_events":sum(bool(x.get("context_trimmed")) for x in generation_rows),
 "protected_prompt_truncation_incidents":sum(bool(x.get("protected_prompt_truncated")) for x in generation_rows),
 "malformed_serialized_subqueries":sum(str(q.get("text","")).lstrip().startswith(("{'query':",'{"query":')) for r in non_oracle for q in r.get("planner",{}).get("retrieval_queries",[])),
 "parser_forced_abstentions":sum(bool(r.get("forced_parser_abstain")) for r in FULL_ROWS),
 "final_infrastructure_failures":sum(bool(r.get("final_decision",{}).get("infrastructure_failure")) for r in FULL_ROWS),
 "abstentions":sum(r["abstain"] for r in FULL_ROWS),"targeted_retries":sum(r.get("retrieval_rounds")==2 for r in FULL_ROWS),
 "official_image_pixel_questions":len({x.get("qid") for x in generation_rows if x.get("qid") in IMAGE_SELECTED_QIDS and x.get("image_count",0)>0}),
}
required_stages=[health[k] for k in ["planner","stage1","stage2","verifier","aggregator"]]
health["pipeline_passed"]=all(v==16 for v in health["questions_completed"].values()) and all(x["total"] and x["parsed"]==x["total"] for x in required_stages) and health["final_infrastructure_failures"]==0 and health["parser_forced_abstentions"]==0 and health["protected_prompt_truncation_incidents"]==0 and health["malformed_serialized_subqueries"]==0

before={"MAMMQA_ORACLE":{"R@5":None,"R@10":None,"EM":.25,"F1":.2638888889,"Abstention":.5625},
        "B1_Dense":{"R@5":.4583333333,"R@10":.5208333333,"EM":0.0,"F1":0.0,"Abstention":.9375},
        "B2_Planner_Dense":{"R@5":.1145833333,"R@10":.1145833333,"EM":0.0,"F1":0.0,"Abstention":1.0},
        "FULL":{"R@5":.1145833333,"R@10":.2083333333,"EM":0.0,"F1":0.0,"Abstention":1.0}}
after={r["Method"]:{k:r[k] for k in ["R@5","R@10","EM","F1","Abstention"]} for r in summary}
before_after=[{"method":m,"metric":metric,"before":before[m][metric],"after":after[m][metric],"delta":None if before[m][metric] is None or pd.isna(after[m][metric]) else after[m][metric]-before[m][metric]} for m in METHODS for metric in ["R@5","R@10","EM","F1","Abstention"]]

traces=[]; trace_rows=[]
def add_trace(row,label):
    if row and row["qid"] not in {x[1]["qid"] for x in trace_rows}:trace_rows.append((label,row))
b1_by={r["qid"]:r for r in ROWS_BY_METHOD["B1_Dense"]}
add_trace(next((r for r in FULL_ROWS if set(map(str,r["gold_support_ids"]))<=all_parent_ids(b1_by[r["qid"]],"retrieval_candidates") and not set(map(str,r["gold_support_ids"]))<=all_parent_ids(r,"retrieval_candidates")),None),"B1 found gold but FULL did not")
add_trace(next((r for r in FULL_ROWS if set(map(str,r["gold_support_ids"]))<=all_parent_ids(r,"retrieval_candidates") and not set(map(str,r["gold_support_ids"]))<=all_parent_ids(r,"context_evidence")),None),"candidate contained gold but context removed it")
add_trace(next((r for r in FULL_ROWS if any(answer_matches(((e or {}).get("parsed_output") or {}).get("candidate_answer",""),r["gold_answers"]) for e in envelopes_for(r,"stage1"))),None),"Stage I produced a correct candidate")
add_trace(next((r for r in FULL_ROWS if r["qid"] in IMAGE_SELECTED_QIDS),None),"image-heavy case")
for r in FULL_ROWS:
    if len(trace_rows)>=4:break
    add_trace(r,"fallback representative case")
for label,r in trace_rows[:4]:
    traces.append({"qid":r["qid"],"question":r["question"],"planner":r["planner"],"routing":r["routing"],
                   "diagnostic_case":label,
                   "candidate_ids":r["candidate_ids"],"context_ids":r["context_ids"],"stage1":r["stage1"],"stage2":r["cross_modal"],
                   "verifier":r["verification"],"feedback":r["targeted_feedback"],"aggregator":r["aggregator_raw"],"final_decision":r["final_decision"],
                   "gold_after_inference":{"answers":r["gold_answers"],"support_ids":r["gold_support_ids"],"exact_match":r["exact_match"]}})

SDF.to_csv(DIRS["metrics"]/"summary.csv",index=False); RDF.to_csv(DIRS["metrics"]/"retrieval_summary.csv",index=False)
PQDF.to_csv(DIRS["metrics"]/"per_question_FULL.csv",index=False); FDF.to_csv(DIRS["metrics"]/"failure_analysis.csv",index=False)
pd.DataFrame(before_after).to_csv(DIRS["metrics"]/"before_after_audit.csv",index=False)
(DIRS["diagnostics"]/"pipeline_health.json").write_text(json.dumps(health,indent=2,allow_nan=True),encoding="utf-8")
(DIRS["metrics"]/"pipeline_health.json").write_text(json.dumps(health,indent=2,allow_nan=True),encoding="utf-8")
(DIRS["diagnostics"]/"manual_traces.json").write_text(json.dumps(traces,indent=2,ensure_ascii=False,default=str),encoding="utf-8")
(DIRS["diagnostics"]/"failure_counts.json").write_text(json.dumps(Counter(FDF["failure_type"]),indent=2),encoding="utf-8")

idx=SDF.set_index("Method"); fs=idx.loc["FULL"]
report=[]
report += ["A. FULL PIPELINE STATUS",f"FULL END-TO-END PIPELINE: {'PASSED' if health['pipeline_passed'] else 'FAILED HEALTH CHECK'}",f"Questions evaluated: {len(FULL_ROWS)}/16",json.dumps(health,indent=2,allow_nan=True)]
if health["pipeline_passed"]:
    report += ["","B. QA RESULT",f"Exact correct: {int(fs.correct_count)} / 16",f"EM / Accuracy: {100*fs.EM:.2f}%",f"Mean F1: {100*fs.F1:.2f}%",f"Abstention: {100*fs.Abstention:.2f}%"]
else:
    report += ["","B. QA RESULT","RUN INVALID FOR QA ACCURACY: one or more structured pipeline health checks failed.","Raw metrics were saved for debugging but must not be interpreted as model performance."]
fr=RDF.set_index("Method").loc["FULL"]; report += ["","C. RETRIEVAL RESULT"]+[f"{k}: {fr[k]:.4f}" for k in ["Recall@1","Recall@3","Recall@5","Recall@10","MRR","candidate_gold_coverage","context_gold_coverage"]]
report += ["","D. CONTROLLED COMPARISON",SDF.to_string(index=False),"","FULL PER-QUESTION TABLE",PQDF.to_string(index=False)]
report += ["","E. FAILURE BREAKDOWN",json.dumps(Counter(FDF["failure_type"]),indent=2),f"Feedback recovered evidence: {sum(FDF.feedback_recovered_evidence)}",f"Feedback corrected answer: {sum(FDF.feedback_corrected_answer)}"]
report += ["","F. CONTROLLED DELTAS",
           f"B2 R@5 - B1 R@5: {100*(idx.loc['B2_Planner_Dense','R@5']-idx.loc['B1_Dense','R@5']):+.2f} points",
           f"FULL R@5 - B1 R@5: {100*(idx.loc['FULL','R@5']-idx.loc['B1_Dense','R@5']):+.2f} points",
           f"B2 EM - B1 EM: {100*(idx.loc['B2_Planner_Dense','EM']-idx.loc['B1_Dense','EM']):+.2f} points",
           f"FULL EM - B1 EM: {100*(idx.loc['FULL','EM']-idx.loc['B1_Dense','EM']):+.2f} points",
           f"MAMMQA_ORACLE EM - FULL EM: {100*(idx.loc['MAMMQA_ORACLE','EM']-idx.loc['FULL','EM']):+.2f} points"]
report += ["","G. FOUR MANUAL DIAGNOSTIC TRACES",json.dumps(traces,indent=2,ensure_ascii=False,default=str)]
report += ["","H. BEFORE/AFTER AUDIT",pd.DataFrame(before_after).to_string(index=False),"","Interpretation: this is a fixed n=16 pipeline-validation smoke test; one exact answer changes EM by 6.25 points. It is not a final benchmark or statistical-significance claim."]
log_text="\n".join(report); (OUTPUT_ROOT/"proper-local-pipeline.log").write_text(log_text,encoding="utf-8")
print(log_text)
print("\nOUTPUT_ROOT =",OUTPUT_ROOT)
print("Repair is pipeline engineering only; no gold answers were used to choose prompts, budgets, thresholds, or settings.")